# Пайплайн предобработки ЭЭГ при ЧМТ и в контрольной группе

## Назначение ноутбука

Данный ноутбук реализует воспроизводимый пайплайн подготовки ЭЭГ-данных для сравнительного анализа пациентов с черепно-мозговой травмой (ЧМТ) и здоровых испытуемых контрольной группы.

В рамках ноутбука выполняются следующие этапы:

1. загрузка уже  локальных файлов ЭЭГ;
2. приведение данных ЧМТ и контрольной группы к единому формату;
3. стандартизация частоты дискретизации, каналов и длительности эпох;
4. базовая предобработка сигнала;
5. Загрузка и базовая стандартизация ЭЭГ
6. Очистка артефактов: плохие каналы, ICA/ICLabel, AutoReject
7. Унификация каналов
8. Финальный QC
9. Итог подготовки данных

Ноутбук не выполняет скачивание данных из OpenNeuro. Предполагается, что контрольные EDF-файлы уже были заранее загружены в локальную папку, путь к которой задан в конфигурации.

## Методическая схема анализа

Исследование построено как сравнительный анализ двух групп:

- **ЧМТ** — пациенты с черепно-мозговой травмой;
- **Контроль** — здоровые испытуемые из открытого датасета OpenNeuro ds005385.

Поскольку данные получены из разных источников и имеют разные исходные форматы, основной задачей данного этапа является их приведение к единому техническому стандарту.

Для этого используются следующие правила:

1. все записи переводятся в формат `mne.Epochs`;
2. анализ проводится на фиксированных 4-секундных эпохах;
3. частота дискретизации приводится к 500 Гц;
4. используется общий набор каналов, доступный в обеих группах;
5. данные фильтруются в диапазоне 0.5–70 Гц;
6. дополнительно применяется notch-фильтр для подавления сетевой помехи;
7. для всех записей используется единая схема референса;
8. результаты сохраняются в единую рабочую директорию `work/output`.

Такой подход позволяет снизить влияние технических различий между источниками данных и подготовить сопоставимое признаковое пространство для дальнейшего статистического анализа и машинного обучения.

## Входные данные

### Данные пациентов с ЧМТ

Данные пациентов с ЧМТ представлены в формате MATLAB `.mat` и содержат структуру EEGLAB `EEG`.

Ожидается, что в файлах могут присутствовать следующие поля:

- `EEG.data` — массив ЭЭГ-данных;
- `EEG.srate` — частота дискретизации;
- `EEG.chanlocs` — информация о каналах;
- `EEG.bad_chans` — предварительно размеченные плохие каналы, если доступны;
- `EEG.bad_epochs` — предварительно размеченные плохие эпохи, если доступны;
- ICA-поля, если независимые компоненты были рассчитаны ранее.

### Контрольные данные

Для контрольной группы используются EDF-файлы из датасета OpenNeuro ds005385.

В данном ноутбуке скачивание данных не выполняется. Предполагается, что EDF-файлы уже находятся в папке:


`data/openneuro/ds005385/` или в другом пути, указанном в параметре:
`CONFIG["ctl_ds005385_root"]`

Для анализа приоритетно используются записи состояния покоя с закрытыми глазами и протоколом acq-pre.


---

##  Ожидаемые результаты ноутбука


## Выходные данные

После выполнения ноутбука формируются следующие результаты:

1. стандартизированные ЭЭГ-записи в формате MNE;
2. таблица inventory с описанием доступных записей;
3. таблицы контроля качества;
4. таблицы признаков на уровне эпох, записей или субъектов;
5. служебные логи выполнения пайплайна;
6. файлы с ошибками, если часть записей не удалось обработать.

Все результаты сохраняются в рабочую директорию:


`work/output/`

Структура выходных данных задаётся автоматически на основе параметров конфигурации.

# Часть 1. Подготовка окружения

In [ ]:
# @title 0.1. Установка зависимостей

# В этой ячейке устанавливаются основные библиотеки,
# необходимые для обработки ЭЭГ, работы с таблицами и расчёта признаков.
#
# Важно:
# - предполагается, что EDF-файлы контрольной группы уже находятся
#   в локальной папке Google Drive.

!pip -q install mne scipy pandas numpy tqdm

In [ ]:
!pip -q install mne mne-icalabel autoreject scipy pandas numpy matplotlib tqdm

In [ ]:
# @title 0.2. Импорт библиотек

# Стандартные библиотеки Python.
# Используются для работы с путями, логами, ошибками и служебными файлами.
import json
import logging
import math
import os
import re
import traceback
import warnings
from datetime import datetime
from pathlib import Path

# Библиотеки для научных вычислений и работы с таблицами.
import numpy as np
import pandas as pd
import scipy.signal as spsig

# Прогресс-бары для длинных циклов обработки записей.
from tqdm import tqdm

# MNE — основная библиотека для работы с ЭЭГ.
import mne
from mne.preprocessing import ICA
from autoreject import AutoReject
from mne.preprocessing import ICA
from mne_icalabel import label_components

# Отключаем лишние предупреждения, чтобы вывод ноутбука был чище.
# При отладке эту строку можно временно закомментировать.
warnings.filterwarnings("ignore")

In [ ]:
# @title 1. Подключение Google Drive

# Google Drive используется как основное хранилище данных и результатов.


from google.colab import drive

drive.mount("/content/drive")

# Часть 2. Конфигурация проекта
В этой ячейке задаются основные параметры пайплайна:

- пути к данным ЧМТ;
- путь к уже скачанным контрольным EDF-файлам;
- рабочая директория для результатов;
- параметры фильтрации и ресемплинга;
- настройки контроля качества;
- параметры ICA;
- режим тестового запуска.

Все ключевые параметры вынесены в словарь `CONFIG`, чтобы избежать дублирования путей и значений в разных частях ноутбука.



In [ ]:
# @title 2.1. Основная конфигурация проекта

# CONFIG — центральный словарь параметров.
# Все пути и основные настройки пайплайна задаются здесь.
#
# Для GitHub пути должны быть шаблонными и относительными.
# Сырые данные НЕ публикуются в репозитории.
#
# Рекомендуемая локальная структура проекта:
#
# eeg-tbi-preprocessing/
# ├── notebooks/
# ├── data/                         # не публикуется, добавить в .gitignore
# │   ├── tbi_mat/
# │   └── openneuro/
# │       └── ds005385/
# ├── work/                         # не публикуется, добавить в .gitignore
# └── outputs_example/              # можно публиковать только обезличенные файлы
#
# Если проект лежит не в текущей папке, можно задать переменную окружения:
#
# import os
# os.environ["EEG_TBI_PROJECT_ROOT"] = "/path/to/project"
#
# В публичном ноутбуке НЕ указывайте личный путь к Google Drive.

PROJECT_ROOT = Path(
    os.environ.get("EEG_TBI_PROJECT_ROOT", ".")
).expanduser()

# Переходим в корень проекта, чтобы все относительные пути работали одинаково.
os.chdir(PROJECT_ROOT)

CONFIG = {
    # ------------------------------------------------------------------
    # Данные пациентов с ЧМТ
    # ------------------------------------------------------------------
    # Папка с .mat-файлами пациентов.
    # Сырые клинические данные не публикуются на GitHub.
    "tbi_base_dir": Path("data/tbi_mat"),

    # Шаблон поиска .mat-файлов внутри папки с данными ЧМТ.
    "tbi_mat_glob": "**/*.mat",

    # ------------------------------------------------------------------
    # Контрольная группа OpenNeuro ds005385
    # ------------------------------------------------------------------
    # Папка, в которой уже лежат скачанные EDF-файлы контрольной группы.
    "ctl_ds005385_root": Path("data/openneuro/ds005385"),

    # Максимальное число контрольных файлов, которые будут выбраны
    # для дальнейшей обработки.
    "ctl_target_n_files": None,

    # Сохранять ли стандартизированные эпохи в формате .fif.
    "save_standardized_epochs": True,

    # ------------------------------------------------------------------
    # Рабочая папка проекта
    # ------------------------------------------------------------------
    # Здесь сохраняются промежуточные результаты.
    # Эту папку НЕ нужно публиковать на GitHub.
    "work_dir": Path("work"),

    # ------------------------------------------------------------------
    # Тестовый режим
    # ------------------------------------------------------------------
    "tbi_test_mode": False,
    "ctl_test_mode": False,

    # Стратегия выбора файлов в тестовом режиме:
    # - "first"  — первые N файлов;
    # - "random" — случайные N файлов;
    # - "slice" — срез по индексам;
    # - "list"  — явно заданный список файлов.
    "test_strategy": "first",

    # Число файлов для тестового запуска.
    "test_n_files": 2,

    # Seed нужен для воспроизводимого случайного выбора файлов.
    "test_seed": 97,

    # Срез файлов для режима "slice".
    "test_slice": (0, 5),

    # Явный список файлов для режима "list".
    "test_file_list": [],

    # ------------------------------------------------------------------
    # Стандартизация сигнала
    # ------------------------------------------------------------------
    "target_sfreq": 500.0,
    "l_freq": 0.5,
    "h_freq": 70.0,
    "notch_freqs": [50, 100, 150, 200],
    "ref": "average",

    # ------------------------------------------------------------------
    # Контроль качества
    # ------------------------------------------------------------------
    "use_premarked_bads": True,
    "drop_epoch_p2p_uv": 350.0,
    "min_epochs_per_record": 10,
    "max_bad_epoch_prop": 0.5,

    # ------------------------------------------------------------------
    # Bad channels
    # ------------------------------------------------------------------
    "run_bad_channel_detection": True,
    "bad_channel_var_z_threshold": 5.0,

    # ------------------------------------------------------------------
    # ICA / ICLabel
    # ------------------------------------------------------------------
    "run_ica": True,
    "ica_fit_l_freq": 1.0,
    "ica_n_components": 20,
    "ica_method": "infomax",
    "ica_fit_params": {"extended": True},
    "ica_max_iter": 512,
    "ica_random_state": 97,

    "iclabel_artifact_labels": [
        "eye blink",
        "heart beat",
        "muscle artifact",
        "line noise",
        "channel noise",
        "other",
    ],

    "iclabel_exclude_threshold": 0.80,

    # ------------------------------------------------------------------
    # AutoReject
    # ------------------------------------------------------------------
    "run_autoreject": True,
    "autoreject_n_interpolate": [1, 2, 4],
    "autoreject_consensus": [0.6, 0.7, 0.8],
    "autoreject_cv": 3,
    "autoreject_random_state": 97,
}

In [ ]:
# @title 2.3.1 Единый стиль визуализаций

import matplotlib.pyplot as plt
import matplotlib.patches as mpatches

# Единые цвета групп во всём проекте.
COL_TBI = "#9B1B30"      # тёмно-красный для ЧМТ
COL_CONTROL = "#1F3F7A"  # тёмно-синий для контроля

GROUP_COLORS = {
    "TBI": COL_TBI,
    "Control": COL_CONTROL,
}

GROUP_LABELS_RU = {
    "TBI": "ЧМТ",
    "Control": "Контроль",
}

plt.rcParams.update(
    {
        "figure.figsize": (12, 7),
        "figure.dpi": 140,
        "savefig.dpi": 300,
        "font.family": "serif",
        "font.size": 12,
        "axes.titlesize": 14,
        "axes.labelsize": 12,
        "xtick.labelsize": 11,
        "ytick.labelsize": 11,
        "legend.fontsize": 11,
        "axes.spines.top": False,
        "axes.spines.right": False,
        "axes.grid": True,
        "grid.alpha": 0.25,
        "grid.linestyle": "--",
    }
)


def save_figure(fig, output_path: Path) -> None:
    """
    Сохраняет рисунок в файл с аккуратными отступами.

    Parameters
    ----------
    fig : matplotlib.figure.Figure
        Объект рисунка.
    output_path : Path
        Путь для сохранения изображения.
    """
    output_path.parent.mkdir(parents=True, exist_ok=True)
    fig.savefig(output_path, bbox_inches="tight")
    print(f"Рисунок сохранён: {output_path}")

In [ ]:
# @title 2.2. Создание структуры выходных папок

# Основная рабочая директория.
WORK_DIR = CONFIG["work_dir"]

# Единая папка для всех результатов.
OUTPUT_DIR = WORK_DIR / "output"

# Словарь с основными выходными директориями.
OUT = {
    "base": OUTPUT_DIR,
    "logs": OUTPUT_DIR / "logs",
    "tables": OUTPUT_DIR / "tables",
    "epochs": OUTPUT_DIR / "epochs",
    "qc": OUTPUT_DIR / "quality_control",
    "errors": OUTPUT_DIR / "errors",
}

# Создаём все папки, если они ещё не существуют.
for out_dir in OUT.values():
    out_dir.mkdir(parents=True, exist_ok=True)

print("Рабочая директория проекта:")
print(WORK_DIR)

print("\nПапка для результатов:")
print(OUTPUT_DIR)

print("\nСозданные выходные папки:")
for name, path in OUT.items():
    print(f"- {name}: {path}")

In [ ]:
# @title Проверка путей к входным данным

def find_files(base_dir: Path, pattern: str) -> list[Path]:
    """
    Находит файлы по заданному шаблону.

    Parameters
    ----------
    base_dir : Path
        Папка, в которой выполняется поиск.
    pattern : str
        Шаблон поиска, например "**/*.mat" или "**/*.edf".

    Returns
    -------
    list[Path]
        Отсортированный список найденных файлов.
    """
    if not base_dir.exists():
        return []

    return sorted(base_dir.glob(pattern))


# Пути к данным.
tbi_base_dir = CONFIG["tbi_base_dir"]
ctl_base_dir = CONFIG["ctl_ds005385_root"]

# Поиск файлов.
tbi_mat_files = find_files(tbi_base_dir, CONFIG["tbi_mat_glob"])
ctl_edf_files = find_files(ctl_base_dir, "**/*.edf")

# Вывод краткой проверки.
print("Проверка входных данных")
print("-" * 40)

print(f"Папка ЧМТ существует: {tbi_base_dir.exists()}")
print(f"Найдено .mat-файлов ЧМТ: {len(tbi_mat_files)}")

print(f"\nПапка контроля существует: {ctl_base_dir.exists()}")
print(f"Найдено EDF-файлов контроля: {len(ctl_edf_files)}")

# Явная проверка, чтобы не продолжать выполнение с пустыми данными.
if len(tbi_mat_files) == 0:
    raise FileNotFoundError(
        "Не найдены .mat-файлы ЧМТ. "
        "Проверьте CONFIG['tbi_base_dir'] и CONFIG['tbi_mat_glob']."
    )

if len(ctl_edf_files) == 0:
    raise FileNotFoundError(
        "Не найдены EDF-файлы контрольной группы. "
        "Проверьте CONFIG['ctl_ds005385_root']."
    )

## 2.4. Логирование выполнения пайплайна


На этом этапе настраивается система логирования выполнения пайплайна.

Логирование используется для фиксации:

- времени запуска анализа;
- основных параметров конфигурации;
- количества найденных файлов;
- этапов обработки каждой записи;
- предупреждений и ошибок;
- путей к сохранённым промежуточным и итоговым файлам.

Это особенно важно при обработке большого числа ЭЭГ-записей, так как отдельные файлы могут иметь повреждённую структуру, неполный набор каналов или отличаться по техническим параметрам.

Ошибки отдельных файлов не должны останавливать весь пайплайн. Вместо этого они сохраняются в отдельную таблицу, чтобы их можно было проанализировать после завершения обработки.


In [ ]:
# @title  Настройка логирования

def setup_logger(log_dir: Path, logger_name: str = "eeg_pipeline") -> logging.Logger:
    """
    Создаёт и настраивает logger для пайплайна обработки ЭЭГ.

    Logger записывает сообщения одновременно:
    1. в консоль Colab;
    2. в текстовый лог-файл в папке output/logs.

    Parameters
    ----------
    log_dir : Path
        Папка для сохранения лог-файлов.
    logger_name : str, optional
        Имя logger, by default "eeg_pipeline".

    Returns
    -------
    logging.Logger
        Настроенный объект logger.
    """
    log_dir.mkdir(parents=True, exist_ok=True)

    timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
    log_path = log_dir / f"{logger_name}_{timestamp}.log"

    logger = logging.getLogger(logger_name)
    logger.setLevel(logging.INFO)

    # Если ячейка запускается повторно, старые handlers нужно удалить,
    # иначе сообщения будут дублироваться в выводе.
    if logger.handlers:
        logger.handlers.clear()

    file_handler = logging.FileHandler(log_path, encoding="utf-8")
    console_handler = logging.StreamHandler()

    log_format = logging.Formatter(
        fmt="%(asctime)s | %(levelname)s | %(message)s",
        datefmt="%Y-%m-%d %H:%M:%S",
    )

    file_handler.setFormatter(log_format)
    console_handler.setFormatter(log_format)

    logger.addHandler(file_handler)
    logger.addHandler(console_handler)

    logger.info("Логирование пайплайна запущено.")
    logger.info("Лог-файл: %s", log_path)

    return logger


logger = setup_logger(OUT["logs"])

logger.info("Рабочая директория: %s", WORK_DIR)
logger.info("Папка результатов: %s", OUTPUT_DIR)
logger.info("Найдено .mat-файлов ЧМТ: %d", len(tbi_mat_files))
logger.info("Найдено EDF-файлов контроля: %d", len(ctl_edf_files))

## 2.5. Сохранение конфигурации запуска

Для обеспечения воспроизводимости текущая конфигурация пайплайна сохраняется в отдельный JSON-файл.

Это позволяет в дальнейшем восстановить параметры конкретного запуска анализа: пути к данным, настройки фильтрации, параметры ICA, режим тестового запуска и другие ключевые значения.

In [ ]:
# @title Сохранение конфигурации запуска

def convert_config_to_jsonable(config: dict) -> dict:
    """
    Преобразует словарь CONFIG в формат, пригодный для сохранения в JSON.

    Объекты Path и другие нестандартные типы преобразуются в строки.

    Parameters
    ----------
    config : dict
        Исходный словарь конфигурации.

    Returns
    -------
    dict
        Словарь, который можно сохранить в JSON.
    """
    jsonable_config = {}

    for key, value in config.items():
        if isinstance(value, Path):
            jsonable_config[key] = str(value)
        elif isinstance(value, tuple):
            jsonable_config[key] = list(value)
        else:
            jsonable_config[key] = value

    return jsonable_config


run_timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
config_path = OUT["logs"] / f"config_{run_timestamp}.json"

with open(config_path, "w", encoding="utf-8") as file:
    json.dump(
        convert_config_to_jsonable(CONFIG),
        file,
        ensure_ascii=False,
        indent=2,
    )

logger.info("Конфигурация запуска сохранена: %s", config_path)

print("Конфигурация запуска сохранена:")
print(config_path)

## 2.6. Служебная таблица ошибок

При обработке ЭЭГ отдельные файлы могут быть пропущены из-за ошибок чтения, неполного набора каналов, повреждённой структуры данных или несовместимого формата.

Для таких случаев создаётся отдельная таблица ошибок. В неё сохраняются:

- группа данных;
- путь к файлу;
- этап обработки, на котором возникла ошибка;
- тип ошибки;
- текст ошибки.

Такой подход позволяет не останавливать весь пайплайн из-за одного проблемного файла и затем отдельно проанализировать причины исключения записей.

In [ ]:
# @title  Служебные функции для регистрации ошибок

processing_errors = []


def register_processing_error(
    group: str,
    file_path: Path,
    stage: str,
    error: Exception,
) -> None:
    """
    Регистрирует ошибку обработки отдельного файла.

    Parameters
    ----------
    group : str
        Группа данных: "TBI" или "Control".
    file_path : Path
        Путь к файлу, при обработке которого возникла ошибка.
    stage : str
        Название этапа обработки.
    error : Exception
        Объект исключения.
    """
    error_record = {
        "group": group,
        "file_path": str(file_path),
        "stage": stage,
        "error_type": type(error).__name__,
        "error_message": str(error),
    }

    processing_errors.append(error_record)

    logger.error(
        "Ошибка обработки | group=%s | stage=%s | file=%s | error=%s",
        group,
        stage,
        file_path,
        str(error),
    )


def save_processing_errors(errors: list[dict], output_path: Path) -> None:
    """
    Сохраняет таблицу ошибок обработки.

    Parameters
    ----------
    errors : list[dict]
        Список зарегистрированных ошибок.
    output_path : Path
        Путь для сохранения CSV-файла.
    """
    if not errors:
        logger.info("Ошибки обработки не зарегистрированы.")
        return

    errors_df = pd.DataFrame(errors)
    errors_df.to_csv(output_path, index=False)

    logger.warning(
        "Таблица ошибок сохранена: %s | число ошибок: %d",
        output_path,
        len(errors_df),
    )

## 2.7. Безопасное выполнение операций

Для обработки большого числа файлов используется принцип безопасного выполнения: если ошибка возникает на одном файле, она регистрируется, но не останавливает весь пайплайн.

Это особенно важно для ЭЭГ-данных, так как отдельные записи могут отличаться по структуре, содержать неполный набор каналов или иметь технические проблемы при чтении.

In [ ]:
# @title Универсальная функция безопасного выполнения

def run_safely(
    func,
    group: str,
    file_path: Path,
    stage: str,
    default=None,
    **kwargs,
):
    """
    Выполняет функцию с перехватом ошибок.

    Если функция завершается с ошибкой, информация об ошибке
    сохраняется в processing_errors, а выполнение пайплайна продолжается.

    Parameters
    ----------
    func : callable
        Функция, которую нужно выполнить.
    group : str
        Группа данных: "TBI" или "Control".
    file_path : Path
        Путь к обрабатываемому файлу.
    stage : str
        Название этапа обработки.
    default : optional
        Значение, которое возвращается при ошибке.
    **kwargs
        Именованные аргументы, передаваемые в func.

    Returns
    -------
    object
        Результат выполнения func или default при ошибке.
    """
    try:
        return func(file_path=file_path, **kwargs)

    except Exception as error:
        register_processing_error(
            group=group,
            file_path=file_path,
            stage=stage,
            error=error,
        )
        return default

## 2.8. Проверка корректности конфигурации

Перед запуском обработки выполняется автоматическая проверка основных параметров конфигурации.

Проверяются:

- существование входных директорий;
- корректность частот фильтрации;
- корректность целевой частоты дискретизации;
- соответствие запрошенного числа контрольных файлов числу реально доступных EDF;
- параметры тестового режима.

Эта проверка позволяет обнаружить технические ошибки до начала длительной обработки данных.

In [ ]:
# @title 2.8. Проверка корректности CONFIG

def validate_config(config: dict) -> None:
    """
    Проверяет корректность основных параметров CONFIG.

    Parameters
    ----------
    config : dict
        Словарь конфигурации пайплайна.

    Raises
    ------
    ValueError
        Если значения параметров некорректны.
    FileNotFoundError
        Если не найдены входные директории.
    """
    if not config["tbi_base_dir"].exists():
        raise FileNotFoundError(
            f"Не найдена папка с данными ЧМТ: {config['tbi_base_dir']}"
        )

    if not config["ctl_ds005385_root"].exists():
        raise FileNotFoundError(
            "Не найдена папка с контрольными EDF-файлами: "
            f"{config['ctl_ds005385_root']}"
        )

    if config["target_sfreq"] <= 0:
        raise ValueError("target_sfreq должна быть положительным числом.")

    if config["l_freq"] >= config["h_freq"]:
        raise ValueError("l_freq должна быть меньше h_freq.")

    # ctl_target_n_files может быть None.
    # None означает: использовать все выбранные контрольные EDF-файлы.
    ctl_target_n_files = config.get("ctl_target_n_files")

    if ctl_target_n_files is not None and ctl_target_n_files <= 0:
        raise ValueError(
            "ctl_target_n_files должно быть положительным числом "
            "или None."
        )

    if (
        ctl_target_n_files is not None
        and ctl_target_n_files > len(ctl_edf_files)
    ):
        logger.warning(
            "Запрошено %d контрольных файлов, но найдено только %d. "
            "Будут использованы все доступные EDF-файлы.",
            ctl_target_n_files,
            len(ctl_edf_files),
        )

    if config["test_n_files"] <= 0:
        raise ValueError("test_n_files должно быть положительным числом.")

    allowed_test_strategies = {"first", "random", "slice", "list"}

    if config["test_strategy"] not in allowed_test_strategies:
        raise ValueError(
            "Недопустимое значение test_strategy. "
            f"Допустимые варианты: {allowed_test_strategies}"
        )

    if config.get("run_ica", False):
        ica_n_components = config.get("ica_n_components")

        if ica_n_components is None:
            raise ValueError("ica_n_components не должен быть None.")

        if isinstance(ica_n_components, int) and ica_n_components <= 0:
            raise ValueError("ica_n_components должно быть положительным.")

        if isinstance(ica_n_components, float) and not 0 < ica_n_components <= 1:
            raise ValueError(
                "Если ica_n_components задан как float, "
                "он должен быть в диапазоне (0, 1]."
            )

    if config.get("run_autoreject", False):
        if config.get("autoreject_cv", 3) <= 0:
            raise ValueError("autoreject_cv должно быть положительным числом.")

    if config.get("max_bad_epoch_prop", 0.5) <= 0:
        raise ValueError("max_bad_epoch_prop должно быть больше 0.")

    if config.get("max_bad_epoch_prop", 0.5) > 1:
        raise ValueError("max_bad_epoch_prop не должно быть больше 1.")


validate_config(CONFIG)

logger.info("Проверка CONFIG успешно завершена.")
print("CONFIG корректен. Можно переходить к построению inventory.")

# Часть 3. Формирование inventory контрольной группы

На этом этапе формируется таблица `inventory` для контрольных EDF-файлов из OpenNeuro ds005385.

Inventory — это служебная таблица, в которой каждая строка соответствует одной найденной EDF-записи. Для каждой записи извлекаются:

- путь к файлу;
- идентификатор субъекта;
- идентификатор сессии;
- состояние записи (`EyesClosed` или `EyesOpen`);
- протокол регистрации (`acq-pre` или `acq-post`);
- имя файла.

Inventory необходим для воспроизводимого выбора контрольных записей и проверки того, какие именно файлы вошли в дальнейший анализ.

In [ ]:
# @title 3.1. Разбор имени EDF-файла OpenNeuro

def parse_openneuro_edf_path(file_path: Path) -> dict:
    """
    Извлекает метаданные из пути к EDF-файлу OpenNeuro.

    Ожидаемый формат имени файла:
    sub-001_ses-1_task-EyesClosed_acq-pre_eeg.edf

    Parameters
    ----------
    file_path : Path
        Путь к EDF-файлу.

    Returns
    -------
    dict
        Словарь с метаданными записи.
    """
    file_name = file_path.name

    subject_match = re.search(r"(sub-[A-Za-z0-9]+)", file_name)
    session_match = re.search(r"(ses-[A-Za-z0-9]+)", file_name)
    task_match = re.search(r"task-([A-Za-z0-9]+)", file_name)
    acq_match = re.search(r"acq-([A-Za-z0-9]+)", file_name)

    subject_id = subject_match.group(1) if subject_match else None
    session_id = session_match.group(1) if session_match else None
    task = task_match.group(1) if task_match else None
    acquisition = acq_match.group(1) if acq_match else None

    return {
        "group": "Control",
        "dataset": "ds005385",
        "subject_id": subject_id,
        "session_id": session_id,
        "task": task,
        "acquisition": acquisition,
        "file_name": file_name,
        "file_path": str(file_path),
    }

In [ ]:
# @title 3.2. Построение inventory для контрольных EDF-файлов

def build_control_inventory(edf_files: list[Path]) -> pd.DataFrame:
    """
    Формирует inventory-таблицу для контрольных EDF-файлов.

    Parameters
    ----------
    edf_files : list[Path]
        Список путей к EDF-файлам контрольной группы.

    Returns
    -------
    pd.DataFrame
        Таблица с метаданными найденных EDF-записей.
    """
    records = []

    for file_path in edf_files:
        record = parse_openneuro_edf_path(file_path)
        records.append(record)

    inventory = pd.DataFrame(records)

    sort_columns = [
        "subject_id",
        "session_id",
        "task",
        "acquisition",
        "file_name",
    ]

    inventory = inventory.sort_values(sort_columns).reset_index(drop=True)

    return inventory


control_inventory = build_control_inventory(ctl_edf_files)

control_inventory_path = OUT["tables"] / "control_inventory_ds005385_local.csv"
control_inventory.to_csv(control_inventory_path, index=False)

logger.info(
    "Inventory контрольной группы сохранён: %s | число записей: %d",
    control_inventory_path,
    len(control_inventory),
)

print("Inventory контрольной группы сформирован.")
print(f"Число EDF-записей: {len(control_inventory)}")
print(f"Число субъектов: {control_inventory['subject_id'].nunique()}")
print(f"Файл сохранён: {control_inventory_path}")

display(control_inventory.head())

## 3.3. Проверка состава контрольных протоколов

После построения inventory необходимо проверить, какие типы записей доступны в контрольной группе.

Для дальнейшего анализа приоритетным является состояние покоя с закрытыми глазами и протоколом `acq-pre`, так как оно лучше всего соответствует условиям регистрации данных ЧМТ.

Если таких записей недостаточно, могут использоваться дополнительные протоколы согласно заранее заданному приоритету.

In [ ]:
# @title Сводка по протоколам контрольной группы

protocol_summary = (
    control_inventory
    .groupby(["task", "acquisition"], dropna=False)
    .agg(
        n_records=("file_path", "count"),
        n_subjects=("subject_id", "nunique"),
    )
    .reset_index()
    .sort_values(["task", "acquisition"])
)

protocol_summary_path = OUT["tables"] / "control_protocol_summary.csv"
protocol_summary.to_csv(protocol_summary_path, index=False)

logger.info("Сводка протоколов контроля сохранена: %s", protocol_summary_path)

print("Сводка по протоколам контрольной группы:")
display(protocol_summary)

## 3.4. Выбор контрольных записей для дальнейшей обработки

На этом этапе из всех доступных EDF-файлов контрольной группы выбирается подмножество записей для дальнейшей обработки.

Выбор выполняется по заранее заданному приоритету протоколов:

1. `EyesClosed`, `acq-pre`;
2. `EyesClosed`, `acq-post`;
3. `EyesOpen`, `acq-pre`;
4. `EyesOpen`, `acq-post`.

Такой подход позволяет сначала использовать наиболее релевантные записи, а затем при необходимости добрать контрольную выборку из менее приоритетных протоколов.

Количество выбранных файлов ограничивается параметром `CONFIG["ctl_target_n_files"]`.


In [ ]:
# @title Выбор контрольных EDF-файлов по приоритету протоколов

CONTROL_PROTOCOL_PRIORITY = [
    ("EyesClosed", "pre"),
    ("EyesClosed", "post"),
    ("EyesOpen", "pre"),
    ("EyesOpen", "post"),
]


def select_control_records(
    inventory: pd.DataFrame,
    protocol_priority: list[tuple[str, str]],
    target_n_files: int | None = None,
) -> pd.DataFrame:
    """
    Выбирает контрольные EDF-записи по приоритету протоколов.

    Если target_n_files=None, используются все записи,
    соответствующие заданным протоколам.

    Parameters
    ----------
    inventory : pd.DataFrame
        Inventory-таблица контрольных EDF-файлов.
    protocol_priority : list[tuple[str, str]]
        Список приоритетов в формате (task, acquisition).
    target_n_files : int | None, optional
        Максимальное число файлов для выбора. Если None, ограничение
        по числу файлов не применяется.

    Returns
    -------
    pd.DataFrame
        Таблица выбранных контрольных записей.
    """
    selected_parts = []

    for priority_rank, (task, acquisition) in enumerate(
        protocol_priority,
        start=1,
    ):
        protocol_records = inventory[
            (inventory["task"] == task)
            & (inventory["acquisition"] == acquisition)
        ].copy()

        protocol_records["protocol_priority"] = priority_rank
        selected_parts.append(protocol_records)

    selected = pd.concat(selected_parts, ignore_index=True)

    selected = selected.sort_values(
        [
            "protocol_priority",
            "subject_id",
            "session_id",
            "file_name",
        ]
    ).reset_index(drop=True)

    if target_n_files is not None and len(selected) > target_n_files:
        selected = selected.iloc[:target_n_files].copy()

    return selected.reset_index(drop=True)


control_selected = select_control_records(
    inventory=control_inventory,
    protocol_priority=CONTROL_PROTOCOL_PRIORITY,
    target_n_files=CONFIG["ctl_target_n_files"],
)

control_selected_path = OUT["tables"] / "control_selected_local_edf.csv"
control_selected.to_csv(control_selected_path, index=False)

logger.info(
    "Выбранные контрольные EDF сохранены: %s | число файлов: %d",
    control_selected_path,
    len(control_selected),
)

print("Выбор контрольных записей завершён.")
print(f"Выбрано EDF-файлов: {len(control_selected)}")
print(f"Число субъектов: {control_selected['subject_id'].nunique()}")
print(f"Файл сохранён: {control_selected_path}")

display(control_selected.head())

### Результат выбора контрольных записей

На данном этапе сформирована таблица выбранных EDF-файлов контрольной группы.

В таблице сохранены:

- идентификатор субъекта;
- сессия;
- состояние регистрации;
- протокол;
- путь к локальному EDF-файлу;
- приоритет протокола.

Эта таблица фиксирует состав контрольной выборки и используется на следующих этапах для загрузки, стандартизации и предобработки ЭЭГ.

Важно, что в данном ноутбуке не выполняется повторное скачивание данных: все выбранные файлы уже находятся в локальной директории проекта.

# Часть 4. Формирование inventory группы ЧМТ

На этом этапе формируется inventory-таблица для локальных `.mat`-файлов пациентов с черепно-мозговой травмой.

Данные пациентов с ЧМТ были предоставлены научным руководителем и получены на базе Института биофизики клетки РАН, г. Пущино. Данные были собраны квалифицированным врачом, обезличены и не распространяются в открытом доступе, так как являются закрытыми исследовательскими данными.

В данном ноутбуке не выполняется скачивание или внешняя загрузка данных ЧМТ. Используются только локальные `.mat`-файлы, путь к которым задан в параметре:

```python
CONFIG["tbi_base_dir"]

In [ ]:
# @title 4.1. Разбор пути к .mat-файлу ЧМТ

def parse_tbi_mat_path(file_path: Path) -> dict:
    """
    Извлекает базовые метаданные из пути к .mat-файлу ЧМТ.

    Поскольку данные ЧМТ являются локальными и могут иметь неоднородные
    имена файлов, функция использует устойчивую стратегию:

    1. record_id формируется из имени файла без расширения;
    2. subject_id пытается извлечься из имени файла;
    3. если subject_id не найден, используется имя родительской папки;
    4. если и это неинформативно, используется record_id.

    Parameters
    ----------
    file_path : Path
        Путь к .mat-файлу ЧМТ.

    Returns
    -------
    dict
        Словарь с базовыми метаданными записи.
    """
    file_stem = file_path.stem
    parent_name = file_path.parent.name

    subject_patterns = [
        r"(sub[-_ ]?\d+)",
        r"(subject[-_ ]?\d+)",
        r"(patient[-_ ]?\d+)",
        r"(pt[-_ ]?\d+)",
        r"(^\d+)",
    ]

    subject_id = None

    for pattern in subject_patterns:
        match = re.search(pattern, file_stem, flags=re.IGNORECASE)
        if match:
            subject_id = match.group(1)
            break

    if subject_id is None and parent_name:
        subject_id = parent_name

    if subject_id is None:
        subject_id = file_stem

    return {
        "group": "TBI",
        "dataset": "local_tbi",
        "subject_id": subject_id,
        "record_id": file_stem,
        "file_name": file_path.name,
        "parent_dir": parent_name,
        "file_size_mb": file_path.stat().st_size / (1024 ** 2),
        "file_path": str(file_path),
    }

In [ ]:
# @title 4.2. Построение inventory для .mat-файлов ЧМТ

def build_tbi_inventory(mat_files: list[Path]) -> pd.DataFrame:
    """
    Формирует inventory-таблицу для локальных .mat-файлов ЧМТ.

    Parameters
    ----------
    mat_files : list[Path]
        Список путей к .mat-файлам ЧМТ.

    Returns
    -------
    pd.DataFrame
        Таблица с базовыми метаданными записей ЧМТ.
    """
    records = []

    for file_path in mat_files:
        record = parse_tbi_mat_path(file_path)
        records.append(record)

    inventory = pd.DataFrame(records)

    inventory = inventory.sort_values(
        ["subject_id", "record_id", "file_name"]
    ).reset_index(drop=True)

    return inventory


tbi_inventory = build_tbi_inventory(tbi_mat_files)

tbi_inventory_path = OUT["tables"] / "tbi_inventory_local_mat.csv"
tbi_inventory.to_csv(tbi_inventory_path, index=False)

logger.info(
    "Inventory группы ЧМТ сохранён: %s | число записей: %d",
    tbi_inventory_path,
    len(tbi_inventory),
)

print("Inventory группы ЧМТ сформирован.")
print(f"Число .mat-файлов: {len(tbi_inventory)}")
print(f"Число субъектов: {tbi_inventory['subject_id'].nunique()}")
print(f"Файл сохранён: {tbi_inventory_path}")

display(tbi_inventory.head())

## 4.3. Проверка состава группы ЧМТ

После построения inventory проверяется соответствие найденных файлов ожидаемой структуре выборки.

В исходном наборе данных ЧМТ ожидается:

- 215 записей;
- 92 субъекта.

Если фактическое число субъектов отличается от ожидаемого, это может означать, что идентификатор субъекта некорректно извлекается из имени файла или структуры папок. В таком случае необходимо уточнить правило формирования `subject_id`.

In [ ]:
# @title Контрольная проверка состава группы ЧМТ

expected_tbi_records = 215
expected_tbi_subjects = 92

actual_tbi_records = len(tbi_inventory)
actual_tbi_subjects = tbi_inventory["subject_id"].nunique()

print("Контрольная проверка состава группы ЧМТ")
print("-" * 50)
print(f"Ожидаемое число записей: {expected_tbi_records}")
print(f"Фактическое число записей: {actual_tbi_records}")
print(f"Ожидаемое число субъектов: {expected_tbi_subjects}")
print(f"Фактическое число субъектов: {actual_tbi_subjects}")

if actual_tbi_records != expected_tbi_records:
    logger.warning(
        "Число записей ЧМТ отличается от ожидаемого: %d вместо %d",
        actual_tbi_records,
        expected_tbi_records,
    )

if actual_tbi_subjects != expected_tbi_subjects:
    logger.warning(
        "Число субъектов ЧМТ отличается от ожидаемого: %d вместо %d",
        actual_tbi_subjects,
        expected_tbi_subjects,
    )

if (
    actual_tbi_records == expected_tbi_records
    and actual_tbi_subjects == expected_tbi_subjects
):
    print("\nСостав группы ЧМТ соответствует ожидаемому.")

In [ ]:
# @title 4.4. Сводка по числу записей на субъекта в группе ЧМТ

tbi_subject_summary = (
    tbi_inventory
    .groupby("subject_id")
    .agg(
        n_records=("record_id", "count"),
        total_size_mb=("file_size_mb", "sum"),
        median_file_size_mb=("file_size_mb", "median"),
    )
    .reset_index()
    .sort_values(["n_records", "subject_id"], ascending=[False, True])
)

tbi_subject_summary_path = OUT["tables"] / "tbi_subject_summary.csv"
tbi_subject_summary.to_csv(tbi_subject_summary_path, index=False)

print("Сводка по числу записей на субъекта в группе ЧМТ:")
display(tbi_subject_summary.head(20))

print("\nОписание распределения числа записей на субъекта:")
display(tbi_subject_summary["n_records"].describe())

## 4.5. Объединённая сводка состава выборки

После построения отдельных inventory для ЧМТ и контрольной группы формируется общая сводка состава данных.

Она используется для быстрой проверки числа субъектов и записей в каждой группе перед запуском предобработки.

In [ ]:
# @title Объединённая сводка состава выборки

sample_summary = pd.DataFrame(
    [
        {
            "group": "TBI",
            "group_ru": "ЧМТ",
            "n_subjects": tbi_inventory["subject_id"].nunique(),
            "n_records": len(tbi_inventory),
        },
        {
            "group": "Control",
            "group_ru": "Контроль",
            "n_subjects": control_selected["subject_id"].nunique(),
            "n_records": len(control_selected),
        },
    ]
)

sample_summary_path = OUT["tables"] / "sample_summary_inventory.csv"
sample_summary.to_csv(sample_summary_path, index=False)

print("Сводка состава выборки:")
display(sample_summary)

logger.info("Сводка состава выборки сохранена: %s", sample_summary_path)

In [ ]:
# @title 4.6. Контрольная проверка ожидаемого состава выборки

expected_sample_summary = {
    "TBI": {
        "n_subjects": 92,
        "n_records": 215,
    },
    "Control": {
        "n_subjects": 72,
        "n_records": 364,
    },
}

actual_sample_summary = {
    row["group"]: {
        "n_subjects": int(row["n_subjects"]),
        "n_records": int(row["n_records"]),
    }
    for _, row in sample_summary.iterrows()
}

print("Контрольная проверка состава выборки")
print("-" * 50)

for group, expected_values in expected_sample_summary.items():
    actual_values = actual_sample_summary[group]

    print(f"\nГруппа: {group}")
    print(
        "Субъекты: "
        f"{actual_values['n_subjects']} / ожидалось {expected_values['n_subjects']}"
    )
    print(
        "Записи: "
        f"{actual_values['n_records']} / ожидалось {expected_values['n_records']}"
    )

    if actual_values != expected_values:
        logger.warning(
            "Состав группы %s отличается от ожидаемого: %s вместо %s",
            group,
            actual_values,
            expected_values,
        )

if actual_sample_summary == expected_sample_summary:
    print("\nСостав обеих групп соответствует ожидаемому.")

In [ ]:
# @title 4.7. Визуальная сводка состава выборки

fig, axes = plt.subplots(1, 2, figsize=(12, 5))

plot_data = sample_summary.copy()
plot_data["color"] = plot_data["group"].map(GROUP_COLORS)
plot_data["group_ru"] = plot_data["group"].map(GROUP_LABELS_RU)

axes[0].bar(
    plot_data["group_ru"],
    plot_data["n_subjects"],
    color=plot_data["color"],
)
axes[0].set_title("Число субъектов")
axes[0].set_ylabel("Количество субъектов")

for index, row in plot_data.iterrows():
    axes[0].text(
        index,
        row["n_subjects"] + 1,
        str(row["n_subjects"]),
        ha="center",
        va="bottom",
    )

axes[1].bar(
    plot_data["group_ru"],
    plot_data["n_records"],
    color=plot_data["color"],
)
axes[1].set_title("Число записей")
axes[1].set_ylabel("Количество записей")

for index, row in plot_data.iterrows():
    axes[1].text(
        index,
        row["n_records"] + 5,
        str(row["n_records"]),
        ha="center",
        va="bottom",
    )

fig.suptitle("Состав исходной выборки", fontsize=16, y=1.03)
fig.tight_layout()

figure_path = OUT["qc"] / "sample_composition_overview.png"
save_figure(fig, figure_path)

plt.show()

### Итог inventory-этапа

В результате inventory-этапа были сформированы таблицы для обеих групп:

- `tbi_inventory_local_mat.csv` — локальные `.mat`-файлы пациентов с ЧМТ;
- `control_inventory_ds005385_local.csv` — все найденные EDF-файлы контрольной группы OpenNeuro ds005385;
- `control_selected_local_edf.csv` — EDF-файлы, передаваемые в дальнейшую обработку;
- `sample_summary_inventory.csv` — общая сводка числа субъектов и записей.

Итоговый состав данных перед предобработкой:

| Группа | Число субъектов | Число записей |
|---|---:|---:|
| ЧМТ | 92 | 215 |
| Контроль | 72 | 364 |

После проверки состава выборки можно переходить к загрузке ЭЭГ и приведению записей к единому формату.

# Часть 5. Загрузка и стандартизация ЭЭГ

После формирования inventory начинается основной этап подготовки сигналов.

Цель этого этапа — привести записи ЧМТ и контрольной группы к единому формату, пригодному для последующего расчёта признаков.

Поскольку исходные данные имеют разные форматы:

- ЧМТ: локальные `.mat`-файлы со структурой EEGLAB;
- Контроль: EDF-файлы OpenNeuro ds005385;

необходимо выполнить унифицированную процедуру загрузки и стандартизации.

На этом этапе для каждой записи выполняются следующие операции:

1. чтение исходного файла;
2. преобразование данных в объект MNE;
3. проверка частоты дискретизации;
4. приведение частоты дискретизации к 500 Гц;
5. унификация названий каналов;
6. фильтрация сигнала;
7. применение notch-фильтра;
8. установка common average reference;
9. сегментация или проверка 4-секундных эпох;
10. сохранение стандартизированных данных и служебной информации.

Итогом этапа является набор сопоставимых ЭЭГ-записей, подготовленных к дальнейшему контролю качества и расчёту признаков.

## 5.1. Общие параметры стандартизации

Для обеих групп используются единые параметры обработки сигнала.

Ключевые настройки берутся из `CONFIG`:

- целевая частота дискретизации — 500 Гц;
- полосовой фильтр — 0.5–70 Гц;
- notch-фильтр — 50 Гц и гармоники;
- референс — common average reference;
- длительность эпохи — 4 секунды.

Единые параметры предобработки необходимы для того, чтобы минимизировать влияние технических различий между источниками данных.

In [ ]:
# @title Общие параметры стандартизации ЭЭГ

TARGET_SFREQ = CONFIG["target_sfreq"]
L_FREQ = CONFIG["l_freq"]
H_FREQ = CONFIG["h_freq"]
NOTCH_FREQS = CONFIG["notch_freqs"]
REFERENCE = CONFIG["ref"]

# Длительность эпохи в секундах.
EPOCH_LENGTH_SEC = 4.0

print("Параметры стандартизации ЭЭГ")
print("-" * 50)
print(f"Целевая частота дискретизации: {TARGET_SFREQ} Гц")
print(f"Полосовой фильтр: {L_FREQ}–{H_FREQ} Гц")
print(f"Notch-фильтр: {NOTCH_FREQS}")
print(f"Референс: {REFERENCE}")
print(f"Длительность эпохи: {EPOCH_LENGTH_SEC} с")

## 5.2. Унификация названий каналов

Перед объединением данных из разных источников необходимо привести названия каналов к единому виду.

В разных форматах один и тот же канал может быть записан по-разному, например:

- `FP1`;
- `Fp1`;
- `EEG Fp1`;
- `Fp1-Ref`.

Для дальнейшего анализа такие названия должны быть стандартизированы. Это необходимо для корректного пересечения каналов между группами, интерполяции плохих каналов и расчёта признаков по ROI.

In [ ]:
# @title Функции унификации названий каналов

def clean_channel_name(channel_name: str) -> str:
    """
    Приводит название ЭЭГ-канала к единому формату.

    Parameters
    ----------
    channel_name : str
        Исходное название канала.

    Returns
    -------
    str
        Стандартизированное название канала.
    """
    cleaned_name = str(channel_name).strip()

    # Удаляем распространённые служебные префиксы и суффиксы.
    cleaned_name = re.sub(r"^EEG\s+", "", cleaned_name, flags=re.IGNORECASE)
    cleaned_name = re.sub(r"-Ref$", "", cleaned_name, flags=re.IGNORECASE)
    cleaned_name = re.sub(r"-REF$", "", cleaned_name, flags=re.IGNORECASE)

    # Убираем лишние пробелы.
    cleaned_name = cleaned_name.replace(" ", "")

    # Приводим типичные названия 10–20 к привычному регистру.
    replacements = {
        "FP1": "Fp1",
        "FP2": "Fp2",
        "FZ": "Fz",
        "CZ": "Cz",
        "PZ": "Pz",
        "OZ": "Oz",
    }

    upper_name = cleaned_name.upper()

    if upper_name in replacements:
        return replacements[upper_name]

    return cleaned_name


def standardize_channel_names(raw: mne.io.BaseRaw) -> mne.io.BaseRaw:
    """
    Стандартизирует названия каналов в объекте MNE Raw.

    Parameters
    ----------
    raw : mne.io.BaseRaw
        Объект Raw с ЭЭГ-сигналом.

    Returns
    -------
    mne.io.BaseRaw
        Raw-объект с обновлёнными названиями каналов.
    """
    rename_mapping = {
        channel_name: clean_channel_name(channel_name)
        for channel_name in raw.ch_names
    }

    raw = raw.copy()
    raw.rename_channels(rename_mapping)

    return raw

## 5.3. Базовая предобработка Raw-сигнала

После загрузки запись приводится к единому виду.

Выполняются следующие операции:

1. стандартизация названий каналов;
2. выбор только ЭЭГ-каналов;
3. установка стандартной схемы расположения электродов;
4. ресемплинг до целевой частоты дискретизации;
5. полосовая фильтрация;
6. notch-фильтрация;
7. применение common average reference.

Эта функция не выполняет ICA и расширенный контроль качества. Эти этапы оформляются отдельно, чтобы предобработка оставалась прозрачной и воспроизводимой.

In [ ]:
# @title Базовая предобработка Raw-сигнала

def preprocess_raw_basic(
    raw: mne.io.BaseRaw,
    target_sfreq: float,
    l_freq: float,
    h_freq: float,
    notch_freqs: list[float],
    reference: str = "average",
) -> mne.io.BaseRaw:
    """
    Выполняет базовую стандартизацию Raw-сигнала.

    Parameters
    ----------
    raw : mne.io.BaseRaw
        Исходный Raw-объект MNE.
    target_sfreq : float
        Целевая частота дискретизации.
    l_freq : float
        Нижняя граница полосового фильтра.
    h_freq : float
        Верхняя граница полосового фильтра.
    notch_freqs : list[float]
        Частоты notch-фильтра.
    reference : str, optional
        Тип референса, by default "average".

    Returns
    -------
    mne.io.BaseRaw
        Предобработанный Raw-объект.
    """
    raw = raw.copy()

    # Приводим названия каналов к единому формату.
    raw = standardize_channel_names(raw)

    # Оставляем только ЭЭГ-каналы, если в записи есть служебные каналы.
    raw.pick_types(eeg=True, exclude=[])

    # Устанавливаем стандартную 10–20/10–10 монтажную схему.
    # on_missing="ignore" позволяет не прерывать обработку,
    # если для части каналов нет координат.
    montage = mne.channels.make_standard_montage("standard_1020")
    raw.set_montage(montage, on_missing="ignore")

    # Ресемплинг выполняется только если частота отличается от целевой.
    current_sfreq = raw.info["sfreq"]

    if not np.isclose(current_sfreq, target_sfreq):
        raw.resample(target_sfreq)

    # Полосовая фильтрация.
    raw.filter(
        l_freq=l_freq,
        h_freq=h_freq,
        fir_design="firwin",
        verbose=False,
    )

    # Notch-фильтр для подавления сетевой помехи.
    available_notch_freqs = [
        freq for freq in notch_freqs
        if freq < raw.info["sfreq"] / 2
    ]

    if available_notch_freqs:
        raw.notch_filter(
            freqs=available_notch_freqs,
            verbose=False,
        )

    # Common average reference.
    if reference == "average":
        raw.set_eeg_reference("average", projection=False, verbose=False)

    return raw

## 5.4. Сегментация на фиксированные эпохи

Для сопоставимости записей обеих групп данные разбиваются на фиксированные 4-секундные эпохи.

Такой формат удобен для последующего расчёта спектральных, временных, событийных и сетевых признаков.

Сегментация выполняется без перекрытия. Каждая эпоха рассматривается как отдельный фрагмент сигнала, но итоговые признаки далее агрегируются на уровне записи или субъекта.

In [ ]:
# @title Сегментация Raw на 4-секундные эпохи

def make_fixed_length_epochs(
    raw: mne.io.BaseRaw,
    epoch_length_sec: float = 4.0,
) -> mne.Epochs:
    """
    Разбивает Raw-сигнал на фиксированные эпохи.

    Parameters
    ----------
    raw : mne.io.BaseRaw
        Предобработанный Raw-объект.
    epoch_length_sec : float, optional
        Длительность эпохи в секундах, by default 4.0.

    Returns
    -------
    mne.Epochs
        Объект Epochs с фиксированными неперекрывающимися эпохами.
    """
    events = mne.make_fixed_length_events(
        raw,
        id=1,
        duration=epoch_length_sec,
    )

    epochs = mne.Epochs(
        raw,
        events,
        event_id={"fixed": 1},
        tmin=0.0,
        tmax=epoch_length_sec - 1.0 / raw.info["sfreq"],
        baseline=None,
        preload=True,
        verbose=False,
    )

    return epochs

## 5.5. Извлечение служебной информации о записи

После загрузки и стандартизации каждой записи сохраняется краткая техническая информация:

- группа;
- идентификатор субъекта;
- идентификатор записи;
- число каналов;
- частота дискретизации;
- число эпох;
- длительность эпохи;
- список каналов.

Эта информация используется для контроля качества и проверки сопоставимости данных между группами.

In [ ]:
# @title Извлечение служебной информации о записи

def summarize_epochs(
    epochs: mne.Epochs,
    group: str,
    subject_id: str,
    record_id: str,
    source_path: str,
) -> dict:
    """
    Формирует краткую техническую сводку по объекту Epochs.

    Parameters
    ----------
    epochs : mne.Epochs
        Объект эпох MNE.
    group : str
        Группа данных.
    subject_id : str
        Идентификатор субъекта.
    record_id : str
        Идентификатор записи.
    source_path : str
        Путь к исходному файлу.

    Returns
    -------
    dict
        Словарь с технической информацией о записи.
    """
    sfreq = float(epochs.info["sfreq"])
    n_epochs = len(epochs)
    n_channels = len(epochs.ch_names)

    epoch_length_sec = (
        epochs.tmax - epochs.tmin + 1.0 / sfreq
    )

    return {
        "group": group,
        "subject_id": subject_id,
        "record_id": record_id,
        "source_path": source_path,
        "n_epochs": n_epochs,
        "n_channels": n_channels,
        "sfreq_hz": sfreq,
        "epoch_len_sec": epoch_length_sec,
        "tmin_sec": epochs.tmin,
        "tmax_sec": epochs.tmax,
        "channels": "|".join(epochs.ch_names),
    }

## 5.6. Загрузка и стандартизация контрольных EDF-файлов

На этом этапе выполняется загрузка EDF-файлов контрольной группы OpenNeuro ds005385.

Каждая запись обрабатывается по единой схеме:

1. чтение локального EDF-файла;
2. преобразование в объект `mne.io.Raw`;
3. стандартизация названий каналов;
4. выбор ЭЭГ-каналов;
5. установка стандартного монтажа;
6. ресемплинг до 500 Гц при необходимости;
7. полосовая фильтрация 0.5–70 Гц;
8. notch-фильтрация;
9. применение common average reference;
10. разбиение на фиксированные 4-секундные эпохи;
11. сохранение технической информации о записи.

В данном ноутбуке скачивание EDF-файлов не выполняется. Все файлы читаются из локальной папки, указанной в `CONFIG["ctl_ds005385_root"]`.

In [ ]:
# @title 5.6.1. Загрузка одной контрольной EDF-записи

def load_control_edf_as_epochs(
    file_path: Path,
    target_sfreq: float,
    l_freq: float,
    h_freq: float,
    notch_freqs: list[float],
    reference: str = "average",
    epoch_length_sec: float = 4.0,
) -> mne.Epochs:
    """
    Загружает EDF-файл контрольной группы и преобразует его в MNE Epochs.

    Parameters
    ----------
    file_path : Path
        Путь к локальному EDF-файлу.
    target_sfreq : float
        Целевая частота дискретизации.
    l_freq : float
        Нижняя граница полосового фильтра.
    h_freq : float
        Верхняя граница полосового фильтра.
    notch_freqs : list[float]
        Частоты notch-фильтра.
    reference : str, optional
        Тип референса, by default "average".
    epoch_length_sec : float, optional
        Длительность фиксированной эпохи, by default 4.0.

    Returns
    -------
    mne.Epochs
        Стандартизированные 4-секундные эпохи.
    """
    raw = mne.io.read_raw_edf(
        file_path,
        preload=True,
        verbose=False,
    )

    raw = preprocess_raw_basic(
        raw=raw,
        target_sfreq=target_sfreq,
        l_freq=l_freq,
        h_freq=h_freq,
        notch_freqs=notch_freqs,
        reference=reference,
    )

    epochs = make_fixed_length_epochs(
        raw=raw,
        epoch_length_sec=epoch_length_sec,
    )

    return epochs

In [ ]:
# @title Тестовая загрузка одной контрольной записи

test_control_row = control_selected.iloc[0]
test_control_path = Path(test_control_row["file_path"])

print("Тестовая контрольная запись")
print("-" * 50)
print(f"Файл: {test_control_path}")
print(f"Субъект: {test_control_row['subject_id']}")
print(f"Протокол: {test_control_row['task']} / {test_control_row['acquisition']}")

test_control_epochs = load_control_edf_as_epochs(
    file_path=test_control_path,
    target_sfreq=TARGET_SFREQ,
    l_freq=L_FREQ,
    h_freq=H_FREQ,
    notch_freqs=NOTCH_FREQS,
    reference=REFERENCE,
    epoch_length_sec=EPOCH_LENGTH_SEC,
)

test_control_summary = summarize_epochs(
    epochs=test_control_epochs,
    group="Control",
    subject_id=test_control_row["subject_id"],
    record_id=test_control_row["file_name"].replace(".edf", ""),
    source_path=str(test_control_path),
)

print("\nРезультат тестовой загрузки:")
for key, value in test_control_summary.items():
    if key != "channels":
        print(f"{key}: {value}")

print("\nПервые 10 каналов:")
print(test_control_epochs.ch_names[:10])

## 5.8. Загрузка и стандартизация данных ЧМТ

На этом этапе выполняется загрузка локальных `.mat`-файлов пациентов с черепно-мозговой травмой.

Исходные данные ЧМТ представлены в формате MATLAB/EEGLAB и содержат структуру `EEG`. В зависимости от файла структура может включать:

- `EEG.data` — массив ЭЭГ-сигнала;
- `EEG.srate` — частоту дискретизации;
- `EEG.chanlocs` — информацию о каналах;
- `EEG.bad_chans` — предварительно размеченные плохие каналы, если они доступны;
- `EEG.bad_epochs` — предварительно размеченные плохие эпохи, если они доступны.

Основная задача этого этапа — преобразовать данные ЧМТ в объект `mne.Epochs`, сопоставимый с контрольной группой.

Для каждой записи выполняются следующие операции:

1. чтение локального `.mat`-файла;
2. извлечение структуры `EEG`;
3. извлечение массива данных, частоты дискретизации и названий каналов;
4. преобразование данных в формат MNE;
5. стандартизация названий каналов;
6. ресемплинг до 500 Гц при необходимости;
7. фильтрация 0.5–70 Гц;
8. notch-фильтрация;
9. применение common average reference;
10. сохранение технической информации о записи.

Для данных ЧМТ notch-фильтрация применяется к массиву эпох через mne.filter.notch_filter, поскольку объект EpochsArray в используемой версии MNE не поддерживает метод .notch_filter() напрямую. После фильтрации объект EpochsArray пересобирается с сохранением метаданных, событий и информации о каналах.

In [ ]:
# @title 5.8. Чтение .mat-файла ЧМТ и извлечение структуры EEG

def load_mat_file(file_path: Path) -> dict:
    """
    Загружает MATLAB .mat-файл.

    Parameters
    ----------
    file_path : Path
        Путь к .mat-файлу.

    Returns
    -------
    dict
        Содержимое .mat-файла.
    """
    import scipy.io

    mat_data = scipy.io.loadmat(
        file_path,
        squeeze_me=True,
        struct_as_record=False,
    )

    return mat_data


def extract_eeg_struct(mat_data: dict):
    """
    Извлекает структуру EEG из загруженного .mat-файла.

    Parameters
    ----------
    mat_data : dict
        Содержимое .mat-файла.

    Returns
    -------
    object
        Структура EEG.

    Raises
    ------
    KeyError
        Если структура EEG не найдена.
    """
    if "EEG" in mat_data:
        return mat_data["EEG"]

    # Иногда ключ может иметь другой регистр.
    for key in mat_data.keys():
        if key.lower() == "eeg":
            return mat_data[key]

    raise KeyError("В .mat-файле не найдена структура EEG.")

In [ ]:
# @title 5.9 Извлечение данных, частоты и каналов из EEGLAB

def get_eeg_field(eeg_struct, field_name: str):
    """
    Извлекает поле из структуры EEG.

    Parameters
    ----------
    eeg_struct : object
        Структура EEG.
    field_name : str
        Название поля.

    Returns
    -------
    object
        Значение поля.

    Raises
    ------
    AttributeError
        Если поле отсутствует.
    """
    if hasattr(eeg_struct, field_name):
        return getattr(eeg_struct, field_name)

    raise AttributeError(f"В структуре EEG отсутствует поле: {field_name}")


def extract_channel_names_from_chanlocs(chanlocs) -> list[str]:
    """
    Извлекает названия каналов из EEG.chanlocs.

    Parameters
    ----------
    chanlocs : object
        Поле EEG.chanlocs.

    Returns
    -------
    list[str]
        Список названий каналов.
    """
    channel_names = []

    if np.ndim(chanlocs) == 0:
        chanlocs = [chanlocs]

    for channel in np.ravel(chanlocs):
        if hasattr(channel, "labels"):
            channel_names.append(str(channel.labels))
        elif isinstance(channel, dict) and "labels" in channel:
            channel_names.append(str(channel["labels"]))
        else:
            channel_names.append(f"EEG{len(channel_names) + 1}")

    return channel_names


def ensure_epochs_shape(data: np.ndarray, n_channels: int) -> np.ndarray:
    """
    Приводит массив ЭЭГ к форме n_epochs × n_channels × n_times.

    Parameters
    ----------
    data : np.ndarray
        Исходный массив ЭЭГ.
    n_channels : int
        Ожидаемое число каналов.

    Returns
    -------
    np.ndarray
        Массив формы n_epochs × n_channels × n_times.

    Raises
    ------
    ValueError
        Если форму массива невозможно интерпретировать.
    """
    data = np.asarray(data)

    if data.ndim == 3:
        # Возможные варианты:
        # 1) n_channels × n_times × n_epochs
        # 2) n_epochs × n_channels × n_times
        # 3) n_channels × n_epochs × n_times

        if data.shape[0] == n_channels:
            data = np.transpose(data, (2, 0, 1))
        elif data.shape[1] == n_channels:
            data = data
        elif data.shape[0] != n_channels and data.shape[1] != n_channels:
            raise ValueError(
                "Не удалось определить ось каналов в 3D-массиве EEG.data."
            )

    elif data.ndim == 2:
        # Если данные непрерывные: n_channels × n_times.
        # Преобразуем к одной эпохе.
        if data.shape[0] == n_channels:
            data = data[np.newaxis, :, :]
        elif data.shape[1] == n_channels:
            data = data.T[np.newaxis, :, :]
        else:
            raise ValueError(
                "Не удалось определить ось каналов в 2D-массиве EEG.data."
            )

    else:
        raise ValueError(
            f"Неподдерживаемая размерность EEG.data: {data.ndim}"
        )

    return data

In [ ]:
# @title 5.10. Преобразование EEGLAB-структуры в MNE Epochs

def convert_eeglab_struct_to_epochs(eeg_struct) -> mne.Epochs:
    """
    Преобразует структуру EEGLAB EEG в объект MNE Epochs.

    Parameters
    ----------
    eeg_struct : object
        Структура EEG из .mat-файла.

    Returns
    -------
    mne.Epochs
        Объект MNE Epochs.
    """
    data = get_eeg_field(eeg_struct, "data")
    sfreq = float(get_eeg_field(eeg_struct, "srate"))
    chanlocs = get_eeg_field(eeg_struct, "chanlocs")

    channel_names = extract_channel_names_from_chanlocs(chanlocs)
    channel_names = [clean_channel_name(name) for name in channel_names]

    data = ensure_epochs_shape(
        data=np.asarray(data),
        n_channels=len(channel_names),
    )

    # MNE ожидает данные в вольтах.
    # Если значения похожи на микровольты, переводим в вольты.
    median_abs_value = np.nanmedian(np.abs(data))

    if median_abs_value > 1e-3:
        data = data * 1e-6

    info = mne.create_info(
        ch_names=channel_names,
        sfreq=sfreq,
        ch_types="eeg",
    )

    epochs = mne.EpochsArray(
        data=data,
        info=info,
        tmin=0.0,
        baseline=None,
        verbose=False,
    )

    montage = mne.channels.make_standard_montage("standard_1020")
    epochs.set_montage(montage, on_missing="ignore")

    return epochs

In [ ]:
# @title 5.11. Базовая предобработка Epochs-сигнала

def preprocess_epochs_basic(
    epochs: mne.Epochs,
    target_sfreq: float,
    l_freq: float,
    h_freq: float,
    notch_freqs: list[float],
    reference: str = "average",
) -> mne.Epochs:
    """
    Выполняет базовую стандартизацию объекта MNE Epochs.

    Parameters
    ----------
    epochs : mne.Epochs
        Исходный объект эпох MNE.
    target_sfreq : float
        Целевая частота дискретизации.
    l_freq : float
        Нижняя граница полосового фильтра.
    h_freq : float
        Верхняя граница полосового фильтра.
    notch_freqs : list[float]
        Частоты notch-фильтра.
    reference : str, optional
        Тип референса, by default "average".

    Returns
    -------
    mne.Epochs
        Предобработанный объект Epochs.
    """
    epochs = epochs.copy()

    rename_mapping = {
        channel_name: clean_channel_name(channel_name)
        for channel_name in epochs.ch_names
    }
    epochs.rename_channels(rename_mapping)

    # Современный вариант выбора только ЭЭГ-каналов.
    epochs.pick("eeg")

    montage = mne.channels.make_standard_montage("standard_1020")
    epochs.set_montage(montage, on_missing="ignore")

    current_sfreq = epochs.info["sfreq"]

    if not np.isclose(current_sfreq, target_sfreq):
        epochs.resample(target_sfreq)

    epochs.filter(
        l_freq=l_freq,
        h_freq=h_freq,
        fir_design="firwin",
        verbose=False,
    )

    available_notch_freqs = [
        freq for freq in notch_freqs
        if freq < epochs.info["sfreq"] / 2
    ]

    if available_notch_freqs:
        data = epochs.get_data(copy=True)

        filtered_data = mne.filter.notch_filter(
            x=data,
            Fs=epochs.info["sfreq"],
            freqs=available_notch_freqs,
            verbose=False,
        )

        epochs = mne.EpochsArray(
            data=filtered_data,
            info=epochs.info.copy(),
            events=epochs.events.copy(),
            event_id=epochs.event_id.copy(),
            tmin=epochs.tmin,
            baseline=None,
            metadata=epochs.metadata,
            verbose=False,
        )

    if reference == "average":
        epochs.set_eeg_reference("average", projection=False, verbose=False)

    return epochs

In [ ]:
# @title 5.12. Загрузка одной .mat-записи ЧМТ

def load_tbi_mat_as_epochs(
    file_path: Path,
    target_sfreq: float,
    l_freq: float,
    h_freq: float,
    notch_freqs: list[float],
    reference: str = "average",
) -> mne.Epochs:
    """
    Загружает .mat-файл ЧМТ и преобразует его в MNE Epochs.

    Parameters
    ----------
    file_path : Path
        Путь к локальному .mat-файлу.
    target_sfreq : float
        Целевая частота дискретизации.
    l_freq : float
        Нижняя граница полосового фильтра.
    h_freq : float
        Верхняя граница полосового фильтра.
    notch_freqs : list[float]
        Частоты notch-фильтра.
    reference : str, optional
        Тип референса, by default "average".

    Returns
    -------
    mne.Epochs
        Стандартизированные эпохи ЭЭГ.
    """
    mat_data = load_mat_file(file_path)
    eeg_struct = extract_eeg_struct(mat_data)

    epochs = convert_eeglab_struct_to_epochs(eeg_struct)

    epochs = preprocess_epochs_basic(
        epochs=epochs,
        target_sfreq=target_sfreq,
        l_freq=l_freq,
        h_freq=h_freq,
        notch_freqs=notch_freqs,
        reference=reference,
    )

    return epochs

# Часть 6. Очистка артефактов

После базовой стандартизации ЭЭГ выполняется автоматизированная очистка данных.

На этом этапе применяются три процедуры:

1. автоматическое выявление каналов с аномально высокой дисперсией;
2. ICA-разложение и автоматическая классификация компонент с использованием ICLabel;
3. AutoReject для локальной интерполяции повреждённых каналов на уровне эпох и исключения чрезмерно артефактных эпох.

Такой подход позволяет снизить влияние глазодвигательных, мышечных, кардиальных и технических артефактов, сохраняя при этом воспроизводимость обработки.

In [ ]:
# @title 6.1. Автоматическое выявление плохих каналов

def detect_bad_channels_by_variance(
    epochs: mne.Epochs,
    z_threshold: float = 5.0,
) -> list[str]:
    """
    Выявляет каналы с аномально высокой дисперсией сигнала.

    Parameters
    ----------
    epochs : mne.Epochs
        Эпохированные ЭЭГ-данные.
    z_threshold : float, optional
        Порог отношения дисперсии канала к медианной дисперсии.

    Returns
    -------
    list[str]
        Список каналов, помеченных как плохие.
    """
    data = epochs.get_data(copy=True)

    # Дисперсия считается по всем эпохам и временным точкам.
    channel_variances = np.var(data, axis=(0, 2))
    median_variance = np.median(channel_variances)

    if median_variance <= 0:
        return []

    bad_mask = channel_variances > z_threshold * median_variance

    bad_channels = [
        channel_name
        for channel_name, is_bad in zip(epochs.ch_names, bad_mask)
        if is_bad
    ]

    return bad_channels


def interpolate_bad_channels(
    epochs: mne.Epochs,
    bad_channels: list[str],
) -> mne.Epochs:
    """
    Интерполирует плохие каналы в объекте Epochs.

    Parameters
    ----------
    epochs : mne.Epochs
        ЭЭГ-эпохи.
    bad_channels : list[str]
        Список плохих каналов.

    Returns
    -------
    mne.Epochs
        Epochs после интерполяции плохих каналов.
    """
    epochs = epochs.copy()

    if not bad_channels:
        return epochs

    epochs.info["bads"] = bad_channels

    montage = mne.channels.make_standard_montage("standard_1020")
    epochs.set_montage(montage, on_missing="ignore")

    epochs.interpolate_bads(
        reset_bads=True,
        verbose=False,
    )

    return epochs

In [ ]:
# @title 6.2. ICA и автоматическая классификация компонент ICLabel

def apply_ica_iclabel_cleaning(
    epochs: mne.Epochs,
    config: dict,
) -> tuple[mne.Epochs, dict]:
    """
    Выполняет ICA-разложение, классификацию компонент через ICLabel
    и удаление артефактных компонент.

    Parameters
    ----------
    epochs : mne.Epochs
        Эпохированные ЭЭГ-данные.
    config : dict
        Конфигурация пайплайна.

    Returns
    -------
    tuple[mne.Epochs, dict]
        Очищенные эпохи и словарь QC-информации.
    """
    epochs_for_ica = epochs.copy()

    # ICLabel рекомендуется применять к ICA,
    # обученной на данных с high-pass около 1 Гц.
    epochs_for_ica.filter(
        l_freq=config.get("ica_fit_l_freq", 1.0),
        h_freq=None,
        fir_design="firwin",
        verbose=False,
    )

    ica = ICA(
        n_components=config.get("ica_n_components", 20),
        method=config.get("ica_method", "infomax"),
        fit_params=config.get("ica_fit_params", {"extended": True}),
        max_iter=config.get("ica_max_iter", 512),
        random_state=config.get("ica_random_state", 97),
    )

    ica.fit(
        epochs_for_ica,
        picks="eeg",
        verbose=False,
    )

    iclabel_result = label_components(
        epochs_for_ica,
        ica,
        method="iclabel",
    )

    labels = iclabel_result["labels"]
    probabilities = iclabel_result["y_pred_proba"]

    artifact_labels = set(
        config.get(
            "iclabel_artifact_labels",
            [
                "eye blink",
                "heart beat",
                "muscle artifact",
                "line noise",
                "channel noise",
                "other",
            ],
        )
    )

    exclude_threshold = config.get("iclabel_exclude_threshold", 0.80)

    exclude_components = []

    for component_index, (label, probability) in enumerate(
        zip(labels, probabilities)
    ):
        if label in artifact_labels and probability >= exclude_threshold:
            exclude_components.append(component_index)

    ica.exclude = exclude_components

    cleaned_epochs = epochs.copy()
    ica.apply(
        cleaned_epochs,
        verbose=False,
    )

    qc_info = {
        "n_ica_components": int(ica.n_components_),
        "n_removed_ica_components": len(exclude_components),
        "removed_ica_components": "|".join(map(str, exclude_components)),
        "iclabel_labels": "|".join(labels),
        "iclabel_probabilities": "|".join(
            f"{float(prob):.4f}" for prob in probabilities
        ),
    }

    return cleaned_epochs, qc_info

In [ ]:
# @title 6.3. AutoReject для очистки эпох

def apply_autoreject_cleaning(
    epochs: mne.Epochs,
    config: dict,
) -> tuple[mne.Epochs, dict]:
    """
    Применяет AutoReject к эпохированным ЭЭГ-данным.

    Parameters
    ----------
    epochs : mne.Epochs
        ЭЭГ-эпохи.
    config : dict
        Конфигурация пайплайна.

    Returns
    -------
    tuple[mne.Epochs, dict]
        Очищенные эпохи и QC-информация.
    """
    n_interpolate = np.array(
        config.get("autoreject_n_interpolate", [1, 2, 4])
    )

    consensus = np.array(
        config.get("autoreject_consensus", [0.6, 0.7, 0.8])
    )

    autoreject = AutoReject(
        n_interpolate=n_interpolate,
        consensus=consensus,
        cv=config.get("autoreject_cv", 3),
        random_state=config.get("autoreject_random_state", 97),
        verbose=False,
    )

    cleaned_epochs, reject_log = autoreject.fit_transform(
        epochs,
        return_log=True,
    )

    bad_epoch_mask = reject_log.bad_epochs

    qc_info = {
        "n_epochs_before_autoreject": len(epochs),
        "n_epochs_after_autoreject": len(cleaned_epochs),
        "n_rejected_epochs_autoreject": int(np.sum(bad_epoch_mask)),
        "prop_rejected_epochs_autoreject": float(np.mean(bad_epoch_mask)),
    }

    return cleaned_epochs, qc_info

AutoReject поддерживает автоматический подбор параметров интерполяции/отклонения для M/EEG-эпох; в его API n_interpolate задаёт варианты числа каналов для интерполяции, а если picks не заданы, решения рассчитываются для типов каналов автоматически.

In [ ]:
# @title 6.4. Полная очистка одной записи

def clean_epochs_artifacts(
    epochs: mne.Epochs,
    config: dict,
) -> tuple[mne.Epochs, dict]:
    """
    Выполняет полный блок автоматической очистки ЭЭГ:
    bad channels, ICA/ICLabel и AutoReject.

    Parameters
    ----------
    epochs : mne.Epochs
        Эпохированные ЭЭГ-данные.
    config : dict
        Конфигурация пайплайна.

    Returns
    -------
    tuple[mne.Epochs, dict]
        Очищенные эпохи и QC-информация.
    """
    qc_info = {}

    if config.get("run_bad_channel_detection", True):
        bad_channels = detect_bad_channels_by_variance(
            epochs=epochs,
            z_threshold=config.get("bad_channel_var_z_threshold", 5.0),
        )

        epochs = interpolate_bad_channels(
            epochs=epochs,
            bad_channels=bad_channels,
        )

        qc_info["n_bad_channels"] = len(bad_channels)
        qc_info["bad_channels"] = "|".join(bad_channels)
    else:
        qc_info["n_bad_channels"] = 0
        qc_info["bad_channels"] = ""

    if config.get("run_ica", True):
        epochs, ica_qc = apply_ica_iclabel_cleaning(
            epochs=epochs,
            config=config,
        )
        qc_info.update(ica_qc)

    if config.get("run_autoreject", True):
        epochs, autoreject_qc = apply_autoreject_cleaning(
            epochs=epochs,
            config=config,
        )
        qc_info.update(autoreject_qc)

    return epochs, qc_info

In [ ]:
# @title 6.5. Служебные функции для сохранения очищенных эпох

import gc


def make_safe_filename(value: str) -> str:
    """
    Преобразует строку в безопасное имя файла.

    Parameters
    ----------
    value : str
        Исходная строка.

    Returns
    -------
    str
        Строка, безопасная для использования в имени файла.
    """
    return re.sub(r"[^A-Za-z0-9_.-]+", "_", str(value))


def save_cleaned_epochs(
    epochs: mne.Epochs,
    output_dir: Path,
    group: str,
    record_id: str,
) -> str:
    """
    Сохраняет очищенные эпохи в формате FIF.

    Parameters
    ----------
    epochs : mne.Epochs
        Очищенные ЭЭГ-эпохи.
    output_dir : Path
        Основная папка для сохранения.
    group : str
        Название группы: TBI или Control.
    record_id : str
        Идентификатор записи.

    Returns
    -------
    str
        Путь к сохранённому FIF-файлу.
    """
    group_output_dir = output_dir / group.lower()
    group_output_dir.mkdir(parents=True, exist_ok=True)

    safe_record_id = make_safe_filename(record_id)
    epochs_path = group_output_dir / f"{safe_record_id}_clean_epo.fif"

    epochs.save(
        epochs_path,
        overwrite=True,
        verbose=False,
    )

    return str(epochs_path)

# Часть 7 Массовая обработка данных

На этом этапе единая процедура загрузки и стандартизации применяется ко всем выбранным EDF-файлам контрольной группы.

Для каждой записи сохраняется техническая информация:

- идентификатор субъекта;
- идентификатор записи;
- число эпох;
- число каналов;
- частота дискретизации;
- длительность эпохи;
- список каналов;
- путь к исходному файлу;
- путь к сохранённому MNE-файлу, если сохранение включено.

Ошибки отдельных файлов регистрируются в служебной таблице и не останавливают выполнение всего пайплайна.

In [ ]:
# @title 7.1. Массовая обработка контрольных EDF-файлов с resume/checkpoint и режимами протоколов

def load_existing_inventory(inventory_path: Path) -> pd.DataFrame:
    """
    Загружает уже сохранённый inventory, если он существует.

    Parameters
    ----------
    inventory_path : Path
        Путь к checkpoint/inventory таблице.

    Returns
    -------
    pd.DataFrame
        Таблица уже обработанных записей.
    """
    if inventory_path.exists():
        existing_inventory = pd.read_csv(inventory_path)
        print(f"Найден существующий checkpoint: {inventory_path}")
        print(f"Уже обработано записей: {len(existing_inventory)}")
        return existing_inventory

    return pd.DataFrame()


def save_processing_checkpoint(
    summaries: list[dict],
    inventory_path: Path,
) -> pd.DataFrame:
    """
    Сохраняет текущий checkpoint обработки.

    Parameters
    ----------
    summaries : list[dict]
        Список сводок по обработанным записям.
    inventory_path : Path
        Путь для сохранения checkpoint.

    Returns
    -------
    pd.DataFrame
        Сохранённая inventory-таблица.
    """
    inventory_path.parent.mkdir(parents=True, exist_ok=True)

    inventory = pd.DataFrame(summaries)

    if len(inventory) > 0 and "record_id" in inventory.columns:
        inventory = (
            inventory
            .drop_duplicates(subset=["record_id"], keep="last")
            .sort_values(["group", "subject_id", "record_id"])
            .reset_index(drop=True)
        )

    inventory.to_csv(inventory_path, index=False)

    return inventory


def save_processing_errors_checkpoint(
    errors: list[dict],
    errors_path: Path,
) -> None:
    """
    Сохраняет ошибки обработки сразу после их появления.

    Parameters
    ----------
    errors : list[dict]
        Список ошибок.
    errors_path : Path
        Путь для сохранения CSV.
    """
    errors_path.parent.mkdir(parents=True, exist_ok=True)

    if len(errors) == 0:
        return

    pd.DataFrame(errors).to_csv(errors_path, index=False)


def get_processed_record_ids(existing_inventory: pd.DataFrame) -> set:
    """
    Возвращает множество уже обработанных record_id.

    Parameters
    ----------
    existing_inventory : pd.DataFrame
        Таблица уже обработанных записей.

    Returns
    -------
    set
        Множество record_id.
    """
    if existing_inventory.empty or "record_id" not in existing_inventory.columns:
        return set()

    return set(existing_inventory["record_id"].astype(str))


def select_control_records_for_processing(
    control_df: pd.DataFrame,
    config: dict,
) -> pd.DataFrame:
    """
    Формирует таблицу контрольных записей для обработки.

    Режимы:
    1. primary_only — только основной протокол;
    2. remaining_protocols — все протоколы, кроме основного;
    3. all — все контрольные протоколы.

    Parameters
    ----------
    control_df : pd.DataFrame
        Исходная таблица control_selected.
    config : dict
        Конфигурация пайплайна.

    Returns
    -------
    pd.DataFrame
        Таблица записей для обработки до исключения уже обработанных.
    """
    selected = control_df.copy()

    mode = config.get("ctl_processing_mode", "primary_only")

    primary_task = config.get("ctl_primary_task", "EyesClosed")
    primary_acquisition = config.get("ctl_primary_acquisition", "pre")

    primary_mask = (
        (selected["task"] == primary_task)
        & (selected["acquisition"] == primary_acquisition)
    )

    if mode == "primary_only":
        selected = selected[primary_mask].copy()

        logger.warning(
            "Режим обработки контроля: только основной протокол %s / %s | "
            "записей: %d",
            primary_task,
            primary_acquisition,
            len(selected),
        )

    elif mode == "remaining_protocols":
        selected = selected[~primary_mask].copy()

        logger.warning(
            "Режим обработки контроля: все протоколы, кроме %s / %s | "
            "записей: %d",
            primary_task,
            primary_acquisition,
            len(selected),
        )

    elif mode == "all":
        logger.warning(
            "Режим обработки контроля: все протоколы | записей: %d",
            len(selected),
        )

    else:
        raise ValueError(
            "Неизвестный ctl_processing_mode. Используй один из вариантов: "
            "'primary_only', 'remaining_protocols', 'all'."
        )

    if config.get("ctl_test_mode", False):
        selected = selected.head(config.get("test_n_files", 2)).copy()

        logger.warning(
            "Контрольная группа обрабатывается в тестовом режиме: %d файлов.",
            len(selected),
        )

    return selected


def process_control_record(
    row: pd.Series,
    output_dir: Path,
    save_epochs: bool = True,
) -> dict:
    """
    Загружает, стандартизирует и очищает одну контрольную EDF-запись.

    Этапы обработки:
    1. загрузка EDF;
    2. базовая стандартизация;
    3. сегментация на 4-секундные эпохи;
    4. выявление и интерполяция плохих каналов;
    5. ICA/ICLabel-очистка;
    6. AutoReject;
    7. сохранение очищенных эпох;
    8. формирование технической сводки.

    Parameters
    ----------
    row : pd.Series
        Строка из control_selected.
    output_dir : Path
        Папка для сохранения очищенных эпох.
    save_epochs : bool, optional
        Сохранять ли очищенные эпохи в FIF, by default True.

    Returns
    -------
    dict
        Техническая и QC-сводка по обработанной записи.
    """
    file_path = Path(row["file_path"])
    subject_id = row["subject_id"]
    record_id = file_path.stem

    epochs = load_control_edf_as_epochs(
        file_path=file_path,
        target_sfreq=TARGET_SFREQ,
        l_freq=L_FREQ,
        h_freq=H_FREQ,
        notch_freqs=NOTCH_FREQS,
        reference=REFERENCE,
        epoch_length_sec=EPOCH_LENGTH_SEC,
    )

    n_epochs_before_cleaning = len(epochs)
    n_channels_before_cleaning = len(epochs.ch_names)

    epochs, cleaning_qc = clean_epochs_artifacts(
        epochs=epochs,
        config=CONFIG,
    )

    summary = summarize_epochs(
        epochs=epochs,
        group="Control",
        subject_id=subject_id,
        record_id=record_id,
        source_path=str(file_path),
    )

    summary.update(cleaning_qc)

    summary["task"] = row.get("task")
    summary["acquisition"] = row.get("acquisition")
    summary["protocol_priority"] = row.get("protocol_priority")
    summary["n_epochs_before_cleaning"] = n_epochs_before_cleaning
    summary["n_channels_before_cleaning"] = n_channels_before_cleaning

    if save_epochs:
        epochs_path = save_cleaned_epochs(
            epochs=epochs,
            output_dir=output_dir,
            group="Control",
            record_id=record_id,
        )
        summary["epochs_path"] = epochs_path
    else:
        summary["epochs_path"] = None

    del epochs
    gc.collect()

    return summary


# ---------------------------------------------------------------------
# РЕЖИМ ОБРАБОТКИ
# ---------------------------------------------------------------------
# Варианты:
# 1. "primary_only" — обработать только основной протокол EyesClosed/pre.
# 2. "remaining_protocols" — обработать все остальные протоколы.
# 3. "all" — обработать все протоколы.
#
# После обработки первого протокола поставь:
# CONFIG["ctl_processing_mode"] = "remaining_protocols"
#
# Для обработки по частям можно оставлять batch_start = 0:
# уже обработанные записи будут исключены, и каждый запуск возьмёт
# следующие N необработанных записей.
# ---------------------------------------------------------------------

CONFIG["ctl_processing_mode"] = "remaining_protocols"

CONFIG["ctl_primary_task"] = "EyesClosed"
CONFIG["ctl_primary_acquisition"] = "pre"

CONFIG["ctl_batch_start"] = 0
CONFIG["ctl_batch_size"] = 30

CONFIG["ctl_test_mode"] = False

# Финальные параметры очистки — одинаковые с основным протоколом.
CONFIG["autoreject_cv"] = 2
CONFIG["autoreject_n_interpolate"] = [1, 2]
CONFIG["autoreject_consensus"] = [0.7, 0.8]
CONFIG["ica_n_components"] = min(CONFIG.get("ica_n_components", 20), 20)


# ---------------------------------------------------------------------
# Пути checkpoint
# ---------------------------------------------------------------------

control_epochs_inventory_path = (
    OUT["tables"] / "control_epochs_inventory_cleaned.csv"
)

processing_errors_path = (
    OUT["errors"] / "processing_errors.csv"
)

existing_control_inventory = load_existing_inventory(
    control_epochs_inventory_path
)

processed_record_ids = get_processed_record_ids(existing_control_inventory)

control_epoch_summaries = []

if not existing_control_inventory.empty:
    control_epoch_summaries = existing_control_inventory.to_dict("records")

save_standardized_epochs = CONFIG.get("save_standardized_epochs", True)

control_records_to_process = select_control_records_for_processing(
    control_df=control_selected,
    config=CONFIG,
)

print("План обработки контрольной группы")
print("-" * 70)
print(f"Режим обработки: {CONFIG['ctl_processing_mode']}")
print(f"Всего записей в выбранном режиме: {len(control_records_to_process)}")
print(f"Уже обработано ранее: {len(processed_record_ids)}")

# ---------------------------------------------------------------------
# Сначала исключаем уже обработанные, потом применяем batch
# ---------------------------------------------------------------------

records_remaining_all = []

for _, row in control_records_to_process.iterrows():
    record_id = Path(row["file_path"]).stem

    if record_id in processed_record_ids:
        continue

    records_remaining_all.append(row)

print(f"Осталось обработать до batch-фильтра: {len(records_remaining_all)}")

batch_start = CONFIG.get("ctl_batch_start", None)
batch_size = CONFIG.get("ctl_batch_size", None)

if batch_start is not None and batch_size is not None:
    records_remaining = records_remaining_all[
        int(batch_start): int(batch_start) + int(batch_size)
    ]

    logger.warning(
        "Batch после исключения уже обработанных: start=%s, size=%s, "
        "записей в текущем запуске=%d",
        batch_start,
        batch_size,
        len(records_remaining),
    )
else:
    records_remaining = records_remaining_all

print(f"Осталось обработать в текущем запуске: {len(records_remaining)}")


# ---------------------------------------------------------------------
# Основной цикл обработки с checkpoint после каждой записи
# ---------------------------------------------------------------------

for row in tqdm(
    records_remaining,
    total=len(records_remaining),
    desc="Обработка контроля",
):
    file_path = Path(row["file_path"])
    record_id = file_path.stem

    try:
        summary = process_control_record(
            row=row,
            output_dir=OUT["epochs"],
            save_epochs=save_standardized_epochs,
        )

        control_epoch_summaries.append(summary)
        processed_record_ids.add(record_id)

        # Ключевой момент: сохраняем прогресс после каждой записи.
        control_epochs_inventory = save_processing_checkpoint(
            summaries=control_epoch_summaries,
            inventory_path=control_epochs_inventory_path,
        )

        logger.info(
            "Контроль обработан: %s | checkpoint записей: %d",
            record_id,
            len(control_epochs_inventory),
        )

    except Exception as error:
        register_processing_error(
            group="Control",
            file_path=file_path,
            stage="control_full_preprocessing",
            error=error,
        )

        save_processing_errors_checkpoint(
            errors=processing_errors,
            errors_path=processing_errors_path,
        )

        logger.exception(
            "Ошибка обработки контрольной записи %s: %s",
            record_id,
            error,
        )

    finally:
        gc.collect()


# ---------------------------------------------------------------------
# Финальное сохранение и сводка
# ---------------------------------------------------------------------

control_epochs_inventory = save_processing_checkpoint(
    summaries=control_epoch_summaries,
    inventory_path=control_epochs_inventory_path,
)

logger.info(
    "Inventory очищенных контрольных эпох сохранён: %s | записей: %d",
    control_epochs_inventory_path,
    len(control_epochs_inventory),
)

print("Обработка контрольной группы завершена.")
print(f"Успешно обработано записей всего: {len(control_epochs_inventory)}")
print(f"Ошибок обработки всего: {len(processing_errors)}")
print(f"Файл сохранён: {control_epochs_inventory_path}")

if len(control_epochs_inventory) > 0:
    display(
        control_epochs_inventory
        .groupby(["task", "acquisition"])
        .agg(
            n_subjects=("subject_id", "nunique"),
            n_records=("record_id", "count"),
            n_epochs=("n_epochs", "sum"),
        )
        .reset_index()
        .sort_values(["task", "acquisition"])
    )

display(control_epochs_inventory.head())

In [ ]:
# @title 7.2. Проверка очищенных контрольных эпох

print("Проверка контрольной группы после очистки")
print("-" * 70)

print(f"Успешно обработано записей: {len(control_epochs_inventory)}")

if len(control_epochs_inventory) > 0:
    print(
        "Число субъектов: "
        f"{control_epochs_inventory['subject_id'].nunique()}"
    )

    print("\nЧастоты дискретизации:")
    display(control_epochs_inventory["sfreq_hz"].value_counts().sort_index())

    print("\nДлительность эпох:")
    display(
        control_epochs_inventory["epoch_len_sec"]
        .round(6)
        .value_counts()
        .sort_index()
    )

    print("\nЧисло каналов:")
    display(
        control_epochs_inventory["n_channels"]
        .value_counts()
        .sort_index()
    )

    print("\nОписание числа эпох после очистки:")
    display(control_epochs_inventory["n_epochs"].describe())

    qc_columns = [
        column for column in [
            "n_bad_channels",
            "n_removed_ica_components",
            "n_epochs_before_autoreject",
            "n_epochs_after_autoreject",
            "n_rejected_epochs_autoreject",
            "prop_rejected_epochs_autoreject",
        ]
        if column in control_epochs_inventory.columns
    ]

    if qc_columns:
        print("\nСводка QC-показателей очистки:")
        display(control_epochs_inventory[qc_columns].describe())

In [ ]:
# @title Проверка контрольных записей с малым числом эпох после очистки

low_epoch_control = control_epochs_inventory[
    control_epochs_inventory["n_epochs"] < 20
].copy()

print(f"Записей контроля с числом эпох < 20: {len(low_epoch_control)}")

display(
    low_epoch_control[
        [
            "group",
            "subject_id",
            "record_id",
            "task",
            "acquisition",
            "n_epochs",
            "n_epochs_before_autoreject",
            "n_epochs_after_autoreject",
            "n_rejected_epochs_autoreject",
            "prop_rejected_epochs_autoreject",
            "epochs_path",
        ]
    ].sort_values("n_epochs")
)

In [ ]:
# @title Проверка контрольных записей с высокой долей отклонённых эпох

HIGH_REJECTION_PROP = 0.30

high_rejection_control = control_epochs_inventory[
    control_epochs_inventory["prop_rejected_epochs_autoreject"]
    > HIGH_REJECTION_PROP
].copy()

print(
    "Контрольных записей с долей отклонённых эпох > "
    f"{HIGH_REJECTION_PROP}: {len(high_rejection_control)}"
)

display(
    high_rejection_control[
        [
            "group",
            "subject_id",
            "record_id",
            "task",
            "acquisition",
            "n_epochs",
            "n_epochs_before_autoreject",
            "n_epochs_after_autoreject",
            "n_rejected_epochs_autoreject",
            "prop_rejected_epochs_autoreject",
            "n_bad_channels",
            "n_removed_ica_components",
            "epochs_path",
        ]
    ].sort_values(
        "prop_rejected_epochs_autoreject",
        ascending=False,
    )
)

In [ ]:
# @title Проверка записей с большим числом плохих каналов

HIGH_BAD_CHANNELS = 12

high_bad_channel_control = control_epochs_inventory[
    control_epochs_inventory["n_bad_channels"] > HIGH_BAD_CHANNELS
].copy()

print(
    "Контрольных записей с числом плохих каналов > "
    f"{HIGH_BAD_CHANNELS}: {len(high_bad_channel_control)}"
)

display(
    high_bad_channel_control[
        [
            "group",
            "subject_id",
            "record_id",
            "task",
            "acquisition",
            "n_epochs",
            "n_bad_channels",
            "n_removed_ica_components",
            "prop_rejected_epochs_autoreject",
            "epochs_path",
        ]
    ].sort_values("n_bad_channels", ascending=False)
)

In [ ]:
# @title Финальный QC контрольной группы после ICA/ICLabel/AutoReject

# ---------------------------------------------------------------------
# Пороговые значения финального record-level QC
# ---------------------------------------------------------------------
MIN_EPOCHS_AFTER_CLEANING = 20
MAX_PROP_REJECTED_AUTOREJECT = 0.50
MAX_BAD_CHANNELS = 16

# Папка для QC-таблиц.
# Используем безопасный вариант: если в OUT нет ключа "quality_control",
# создаём папку quality_control рядом с tables.
QC_DIR = OUT.get("quality_control", OUT["tables"].parent / "quality_control")
QC_DIR.mkdir(parents=True, exist_ok=True)

# ---------------------------------------------------------------------
# Расчёт QC-флагов
# ---------------------------------------------------------------------
control_epochs_inventory = control_epochs_inventory.copy()

control_epochs_inventory["qc_pass_cleaning"] = (
    (control_epochs_inventory["n_epochs"] >= MIN_EPOCHS_AFTER_CLEANING)
    & (
        control_epochs_inventory["prop_rejected_epochs_autoreject"]
        <= MAX_PROP_REJECTED_AUTOREJECT
    )
    & (control_epochs_inventory["n_bad_channels"] <= MAX_BAD_CHANNELS)
)

control_epochs_inventory["qc_exclusion_reason"] = ""

control_epochs_inventory.loc[
    control_epochs_inventory["n_epochs"] < MIN_EPOCHS_AFTER_CLEANING,
    "qc_exclusion_reason",
] += "low_n_epochs;"

control_epochs_inventory.loc[
    control_epochs_inventory["prop_rejected_epochs_autoreject"]
    > MAX_PROP_REJECTED_AUTOREJECT,
    "qc_exclusion_reason",
] += "high_autoreject_rejection;"

control_epochs_inventory.loc[
    control_epochs_inventory["n_bad_channels"] > MAX_BAD_CHANNELS,
    "qc_exclusion_reason",
] += "many_bad_channels;"

# ---------------------------------------------------------------------
# Формирование итоговых таблиц
# ---------------------------------------------------------------------
control_excluded_after_cleaning = control_epochs_inventory[
    ~control_epochs_inventory["qc_pass_cleaning"]
].copy()

control_ready_after_cleaning = control_epochs_inventory[
    control_epochs_inventory["qc_pass_cleaning"]
].copy()

control_qc_summary = (
    control_epochs_inventory
    .groupby(["group", "qc_pass_cleaning"])
    .agg(
        n_subjects=("subject_id", "nunique"),
        n_records=("record_id", "count"),
        n_epochs=("n_epochs", "sum"),
        median_epochs=("n_epochs", "median"),
        min_epochs=("n_epochs", "min"),
        max_epochs=("n_epochs", "max"),
        median_bad_channels=("n_bad_channels", "median"),
        max_bad_channels=("n_bad_channels", "max"),
        median_removed_ica=("n_removed_ica_components", "median"),
        max_removed_ica=("n_removed_ica_components", "max"),
        median_prop_rejected_ar=(
            "prop_rejected_epochs_autoreject",
            "median",
        ),
        max_prop_rejected_ar=(
            "prop_rejected_epochs_autoreject",
            "max",
        ),
    )
    .reset_index()
)

# ---------------------------------------------------------------------
# Вывод на экран
# ---------------------------------------------------------------------
print("Финальный QC контрольной группы")
print("-" * 70)
print(f"Всего записей контроля: {len(control_epochs_inventory)}")
print(f"Прошли QC: {len(control_ready_after_cleaning)}")
print(f"Исключены: {len(control_excluded_after_cleaning)}")

display(control_qc_summary)

if len(control_excluded_after_cleaning) > 0:
    excluded_columns = [
        "group",
        "subject_id",
        "record_id",
        "task",
        "acquisition",
        "n_epochs",
        "n_epochs_before_autoreject",
        "n_epochs_after_autoreject",
        "n_rejected_epochs_autoreject",
        "prop_rejected_epochs_autoreject",
        "n_bad_channels",
        "n_removed_ica_components",
        "qc_exclusion_reason",
        "epochs_path",
    ]

    existing_excluded_columns = [
        column for column in excluded_columns
        if column in control_excluded_after_cleaning.columns
    ]

    display(
        control_excluded_after_cleaning[existing_excluded_columns]
        .sort_values(
            [
                "prop_rejected_epochs_autoreject",
                "n_epochs",
            ],
            ascending=[False, True],
        )
    )

# ---------------------------------------------------------------------
# Сохранение файлов
# ---------------------------------------------------------------------
control_inventory_with_qc_path = (
    OUT["tables"] / "control_epochs_inventory_cleaned_with_qc.csv"
)

control_analysis_ready_path = (
    OUT["tables"] / "control_epochs_inventory_analysis_ready.csv"
)

control_excluded_path = (
    QC_DIR / "control_excluded_after_cleaning_qc.csv"
)

control_qc_summary_path = (
    QC_DIR / "control_cleaning_qc_summary.csv"
)

control_epochs_inventory.to_csv(
    control_inventory_with_qc_path,
    index=False,
)

control_ready_after_cleaning.to_csv(
    control_analysis_ready_path,
    index=False,
)

control_excluded_after_cleaning.to_csv(
    control_excluded_path,
    index=False,
)

control_qc_summary.to_csv(
    control_qc_summary_path,
    index=False,
)

# ---------------------------------------------------------------------
# Финальное сообщение
# ---------------------------------------------------------------------
print("QC-файлы контроля сохранены.")
print("Полный inventory с QC:")
print(control_inventory_with_qc_path)

print("\nAnalysis-ready контроль:")
print(control_analysis_ready_path)

print("\nИсключённые записи:")
print(control_excluded_path)

print("\nQC summary:")
print(control_qc_summary_path)

In [ ]:
# @title Исправление checkpoint ЧМТ и тест одной записи после исправления

# ---------------------------------------------------------------------
# 1. Устойчивая загрузка checkpoint/inventory
# ---------------------------------------------------------------------
def load_existing_inventory(inventory_path: Path) -> pd.DataFrame:
    """
    Загружает уже сохранённый inventory, если он существует.

    Если checkpoint существует, но пустой или повреждён,
    функция возвращает пустой DataFrame, чтобы обработку можно было
    начать заново без падения на EmptyDataError.
    """
    if not inventory_path.exists():
        return pd.DataFrame()

    try:
        existing_inventory = pd.read_csv(inventory_path)
    except pd.errors.EmptyDataError:
        logger.warning(
            "Checkpoint существует, но пустой: %s. "
            "Он будет проигнорирован.",
            inventory_path,
        )
        return pd.DataFrame()

    if existing_inventory.empty:
        logger.warning(
            "Checkpoint существует, но не содержит строк: %s. "
            "Он будет проигнорирован.",
            inventory_path,
        )
        return pd.DataFrame()

    print(f"Найден существующий checkpoint: {inventory_path}")
    print(f"Уже обработано записей: {len(existing_inventory)}")

    return existing_inventory


# ---------------------------------------------------------------------
# 2. Проверка и удаление пустого checkpoint ЧМТ
# ---------------------------------------------------------------------
tbi_epochs_inventory_path = (
    OUT["tables"] / "tbi_epochs_inventory_cleaned.csv"
)

if tbi_epochs_inventory_path.exists():
    try:
        tmp_tbi_checkpoint = pd.read_csv(tbi_epochs_inventory_path)

        print("Checkpoint ЧМТ читается:")
        print(tbi_epochs_inventory_path)
        print(f"Форма таблицы: {tmp_tbi_checkpoint.shape}")

        if tmp_tbi_checkpoint.empty:
            print("Checkpoint пустой по строкам, но содержит колонки.")
            print("Оставляем файл: он не должен мешать обработке.")

    except pd.errors.EmptyDataError:
        tbi_epochs_inventory_path.unlink()

        print("Пустой checkpoint ЧМТ удалён:")
        print(tbi_epochs_inventory_path)
else:
    print("Checkpoint ЧМТ пока не существует. Это нормально.")


# ---------------------------------------------------------------------
# 3. Явно задаём таблицу ЧМТ для массовой обработки
# ---------------------------------------------------------------------
tbi_selected = tbi_inventory.copy()

print("\nТаблица ЧМТ для обработки задана:")
print(f"Записей: {len(tbi_selected)}")
display(tbi_selected.head())


# ---------------------------------------------------------------------
# 4. Исправленная функция обработки одной записи ЧМТ
# ---------------------------------------------------------------------
def process_tbi_record(
    row: pd.Series,
    output_dir: Path,
    save_epochs: bool = True,
) -> dict:
    """
    Загружает, стандартизирует и очищает одну .mat-запись ЧМТ.

    ВАЖНО:
    В load_tbi_mat_as_epochs() не передаётся epoch_length_sec,
    потому что текущая версия этой функции не принимает такой аргумент.
    """
    file_path = Path(row["file_path"])
    subject_id = infer_tbi_subject_id(row)
    record_id = infer_tbi_record_id(row)

    epochs = load_tbi_mat_as_epochs(
        file_path=file_path,
        target_sfreq=TARGET_SFREQ,
        l_freq=L_FREQ,
        h_freq=H_FREQ,
        notch_freqs=NOTCH_FREQS,
        reference=REFERENCE,
    )

    n_epochs_before_cleaning = len(epochs)
    n_channels_before_cleaning = len(epochs.ch_names)

    epochs, cleaning_qc = clean_epochs_artifacts(
        epochs=epochs,
        config=CONFIG,
    )

    summary = summarize_epochs(
        epochs=epochs,
        group="TBI",
        subject_id=subject_id,
        record_id=record_id,
        source_path=str(file_path),
    )

    summary.update(cleaning_qc)

    summary["n_epochs_before_cleaning"] = n_epochs_before_cleaning
    summary["n_channels_before_cleaning"] = n_channels_before_cleaning

    for column in [
        "dataset",
        "file_name",
        "parent_dir",
        "file_size_mb",
        "session",
        "condition",
        "task",
        "acquisition",
        "protocol",
        "protocol_priority",
    ]:
        if column in row.index:
            summary[column] = row.get(column)

    if save_epochs:
        epochs_path = save_cleaned_epochs(
            epochs=epochs,
            output_dir=output_dir,
            group="TBI",
            record_id=record_id,
        )
        summary["epochs_path"] = epochs_path
    else:
        summary["epochs_path"] = None

    del epochs
    gc.collect()

    return summary



In [ ]:
# @title 7.3. Массовая обработка ЧМТ .mat-файлов с resume/checkpoint

def select_tbi_records_for_processing(
    tbi_df: pd.DataFrame,
    config: dict,
) -> pd.DataFrame:
    """
    Формирует таблицу записей ЧМТ для обработки.

    Parameters
    ----------
    tbi_df : pd.DataFrame
        Исходная таблица tbi_selected или tbi_inventory.
    config : dict
        Конфигурация пайплайна.

    Returns
    -------
    pd.DataFrame
        Таблица записей для обработки.
    """
    selected = tbi_df.copy()

    # Batch-режим нужен, если Colab часто отключается.
    batch_start = config.get("tbi_batch_start", None)
    batch_size = config.get("tbi_batch_size", None)

    if batch_start is not None and batch_size is not None:
        selected = selected.iloc[
            int(batch_start): int(batch_start) + int(batch_size)
        ].copy()

        logger.warning(
            "Включён batch-режим ЧМТ: start=%s, size=%s, записей=%d",
            batch_start,
            batch_size,
            len(selected),
        )

    if config.get("tbi_test_mode", False):
        selected = selected.head(config.get("test_n_files", 2)).copy()

        logger.warning(
            "Группа ЧМТ обрабатывается в тестовом режиме: %d файлов.",
            len(selected),
        )

    return selected


def infer_tbi_record_id(row: pd.Series) -> str:
    """
    Возвращает record_id для записи ЧМТ.

    Parameters
    ----------
    row : pd.Series
        Строка inventory.

    Returns
    -------
    str
        record_id.
    """
    if "record_id" in row.index and pd.notna(row["record_id"]):
        return str(row["record_id"])

    return Path(row["file_path"]).stem


def infer_tbi_subject_id(row: pd.Series) -> str:
    """
    Возвращает subject_id для записи ЧМТ.

    Parameters
    ----------
    row : pd.Series
        Строка inventory.

    Returns
    -------
    str
        subject_id.
    """
    if "subject_id" in row.index and pd.notna(row["subject_id"]):
        return str(row["subject_id"])

    record_id = infer_tbi_record_id(row)

    # Для формата вроде 3001_1_Rest субъект — первая часть.
    return str(record_id).split("_")[0]


def process_tbi_record(
    row: pd.Series,
    output_dir: Path,
    save_epochs: bool = True,
) -> dict:
    """
    Загружает, стандартизирует и очищает одну .mat-запись ЧМТ.

    Этапы обработки:
    1. загрузка MATLAB/EEGLAB .mat;
    2. базовая стандартизация;
    3. проверка/формирование 4-секундных эпох;
    4. выявление и интерполяция плохих каналов;
    5. ICA/ICLabel-очистка;
    6. AutoReject;
    7. сохранение очищенных эпох;
    8. формирование технической сводки.

    Parameters
    ----------
    row : pd.Series
        Строка из tbi_selected или tbi_inventory.
    output_dir : Path
        Папка для сохранения очищенных эпох.
    save_epochs : bool, optional
        Сохранять ли очищенные эпохи в FIF, by default True.

    Returns
    -------
    dict
        Техническая и QC-сводка по обработанной записи.
    """

    file_path = Path(row["file_path"])
    subject_id = infer_tbi_subject_id(row)
    record_id = infer_tbi_record_id(row)

    epochs = load_tbi_mat_as_epochs(
        file_path=file_path,
        target_sfreq=TARGET_SFREQ,
        l_freq=L_FREQ,
        h_freq=H_FREQ,
        notch_freqs=NOTCH_FREQS,
        reference=REFERENCE,
    )

    n_epochs_before_cleaning = len(epochs)
    n_channels_before_cleaning = len(epochs.ch_names)

    epochs, cleaning_qc = clean_epochs_artifacts(
        epochs=epochs,
        config=CONFIG,
    )

    summary = summarize_epochs(
        epochs=epochs,
        group="TBI",
        subject_id=subject_id,
        record_id=record_id,
        source_path=str(file_path),
    )

    summary.update(cleaning_qc)

    summary["n_epochs_before_cleaning"] = n_epochs_before_cleaning
    summary["n_channels_before_cleaning"] = n_channels_before_cleaning

    for column in [
        "dataset",
        "file_name",
        "parent_dir",
        "file_size_mb",
        "session",
        "condition",
        "task",
        "acquisition",
        "protocol",
        "protocol_priority",
    ]:
        if column in row.index:
            summary[column] = row.get(column)

    if save_epochs:
        epochs_path = save_cleaned_epochs(
            epochs=epochs,
            output_dir=output_dir,
            group="TBI",
            record_id=record_id,
        )
        summary["epochs_path"] = epochs_path
    else:
        summary["epochs_path"] = None

    del epochs
    gc.collect()

    return summary


# ---------------------------------------------------------------------
# Настройки ускорения и устойчивости — те же, что для контроля
# ---------------------------------------------------------------------

# Batch-режим. Можно обрабатывать по 20–40 файлов за запуск.
# Если хочешь обработать всё сразу, оставь None.
CONFIG["tbi_batch_start"] = None
CONFIG["tbi_batch_size"] = None

# Тестовый режим для быстрой проверки.
CONFIG["tbi_test_mode"] = False

# Финальные облегчённые параметры AutoReject.
CONFIG["autoreject_cv"] = 2
CONFIG["autoreject_n_interpolate"] = [1, 2]
CONFIG["autoreject_consensus"] = [0.7, 0.8]

# ICA ограничиваем максимум 20 компонентами.
CONFIG["ica_n_components"] = min(CONFIG.get("ica_n_components", 20), 20)

# ---------------------------------------------------------------------
# Пути checkpoint
# ---------------------------------------------------------------------

tbi_epochs_inventory_path = (
    OUT["tables"] / "tbi_epochs_inventory_cleaned.csv"
)

processing_errors_path = (
    OUT["errors"] / "processing_errors.csv"
)

existing_tbi_inventory = load_existing_inventory(
    tbi_epochs_inventory_path
)

processed_record_ids = get_processed_record_ids(existing_tbi_inventory)

tbi_epoch_summaries = []

if not existing_tbi_inventory.empty:
    tbi_epoch_summaries = existing_tbi_inventory.to_dict("records")

save_standardized_epochs = CONFIG.get("save_standardized_epochs", True)


tbi_records_to_process = select_tbi_records_for_processing(
    tbi_df=tbi_inventory,
    config=CONFIG,
)

print("План обработки группы ЧМТ")
print("-" * 70)
print(f"Всего записей в выбранном наборе: {len(tbi_records_to_process)}")
print(f"Уже обработано ранее: {len(processed_record_ids)}")

records_remaining = []

for _, row in tbi_records_to_process.iterrows():
    record_id = infer_tbi_record_id(row)

    if record_id in processed_record_ids:
        continue

    records_remaining.append(row)

print(f"Осталось обработать: {len(records_remaining)}")

# ---------------------------------------------------------------------
# Основной цикл обработки с checkpoint после каждой записи
# ---------------------------------------------------------------------

for row in tqdm(
    records_remaining,
    total=len(records_remaining),
    desc="Обработка ЧМТ",
):
    file_path = Path(row["file_path"])
    record_id = infer_tbi_record_id(row)

    try:
        summary = process_tbi_record(
            row=row,
            output_dir=OUT["epochs"],
            save_epochs=save_standardized_epochs,
        )

        tbi_epoch_summaries.append(summary)
        processed_record_ids.add(record_id)

        # Ключевой момент: сохраняем прогресс после каждой записи.
        tbi_epochs_inventory = save_processing_checkpoint(
            summaries=tbi_epoch_summaries,
            inventory_path=tbi_epochs_inventory_path,
        )

        logger.info(
            "ЧМТ обработана: %s | checkpoint записей: %d",
            record_id,
            len(tbi_epochs_inventory),
        )

    except Exception as error:
        register_processing_error(
            group="TBI",
            file_path=file_path,
            stage="tbi_full_preprocessing",
            error=error,
        )

        save_processing_errors_checkpoint(
            errors=processing_errors,
            errors_path=processing_errors_path,
        )

        logger.exception(
            "Ошибка обработки записи ЧМТ %s: %s",
            record_id,
            error,
        )

    finally:
        gc.collect()

# Финальное сохранение.
tbi_epochs_inventory = save_processing_checkpoint(
    summaries=tbi_epoch_summaries,
    inventory_path=tbi_epochs_inventory_path,
)

logger.info(
    "Inventory очищенных эпох ЧМТ сохранён: %s | записей: %d",
    tbi_epochs_inventory_path,
    len(tbi_epochs_inventory),
)

print("Обработка группы ЧМТ завершена.")
print(f"Успешно обработано записей: {len(tbi_epochs_inventory)}")
print(f"Ошибок обработки всего: {len(processing_errors)}")
print(f"Файл сохранён: {tbi_epochs_inventory_path}")

display(tbi_epochs_inventory.head())

In [ ]:
# @title 7.4. Проверка очищенных эпох ЧМТ

print("Проверка группы ЧМТ после очистки")
print("-" * 70)

print(f"Успешно обработано записей: {len(tbi_epochs_inventory)}")

if len(tbi_epochs_inventory) > 0:
    print(
        "Число субъектов: "
        f"{tbi_epochs_inventory['subject_id'].nunique()}"
    )

    print("\nЧастоты дискретизации:")
    display(tbi_epochs_inventory["sfreq_hz"].value_counts().sort_index())

    print("\nДлительность эпох:")
    display(
        tbi_epochs_inventory["epoch_len_sec"]
        .round(6)
        .value_counts()
        .sort_index()
    )

    print("\nЧисло каналов:")
    display(
        tbi_epochs_inventory["n_channels"]
        .value_counts()
        .sort_index()
    )

    print("\nОписание числа эпох после очистки:")
    display(tbi_epochs_inventory["n_epochs"].describe())

    qc_columns = [
        column for column in [
            "n_bad_channels",
            "n_removed_ica_components",
            "n_epochs_before_autoreject",
            "n_epochs_after_autoreject",
            "n_rejected_epochs_autoreject",
            "prop_rejected_epochs_autoreject",
        ]
        if column in tbi_epochs_inventory.columns
    ]

    if qc_columns:
        print("\nСводка QC-показателей очистки:")
        display(tbi_epochs_inventory[qc_columns].describe())

In [ ]:
# @title 7.5. Объединение inventory очищенных эпох

standardized_inventory = pd.concat(
    [
        tbi_epochs_inventory,
        control_epochs_inventory,
    ],
    ignore_index=True,
)

standardized_inventory_path = (
    OUT["tables"] / "inventory_epochs_cleaned.csv"
)

standardized_inventory.to_csv(
    standardized_inventory_path,
    index=False,
)

print("Объединённый inventory очищенных эпох:")
print(f"Всего записей: {len(standardized_inventory)}")
print(f"Файл сохранён: {standardized_inventory_path}")

display(
    standardized_inventory
    .groupby("group")
    .agg(
        n_subjects=("subject_id", "nunique"),
        n_records=("record_id", "count"),
        n_epochs=("n_epochs", "sum"),
        median_epochs_per_record=("n_epochs", "median"),
        median_channels=("n_channels", "median"),
        median_sfreq=("sfreq_hz", "median"),
        median_epoch_len_sec=("epoch_len_sec", "median"),
    )
    .reset_index()
)

In [ ]:
# @title 7.6. Сводка ICA/ICLabel/AutoReject по группам

cleaning_summary_columns = [
    "n_bad_channels",
    "n_removed_ica_components",
    "n_epochs_before_autoreject",
    "n_epochs_after_autoreject",
    "n_rejected_epochs_autoreject",
    "prop_rejected_epochs_autoreject",
]

available_cleaning_columns = [
    column for column in cleaning_summary_columns
    if column in standardized_inventory.columns
]

if available_cleaning_columns:
    cleaning_summary = (
        standardized_inventory
        .groupby("group")
        .agg(
            n_records=("record_id", "count"),
            median_bad_channels=("n_bad_channels", "median"),
            max_bad_channels=("n_bad_channels", "max"),
            median_removed_ica_components=(
                "n_removed_ica_components",
                "median",
            ),
            max_removed_ica_components=(
                "n_removed_ica_components",
                "max",
            ),
            median_rejected_epochs_autoreject=(
                "n_rejected_epochs_autoreject",
                "median",
            ),
            median_prop_rejected_epochs_autoreject=(
                "prop_rejected_epochs_autoreject",
                "median",
            ),
            max_prop_rejected_epochs_autoreject=(
                "prop_rejected_epochs_autoreject",
                "max",
            ),
        )
        .reset_index()
    )

    cleaning_summary_path = (
        OUT["tables"] / "cleaning_summary_by_group.csv"
    )

    cleaning_summary.to_csv(cleaning_summary_path, index=False)

    print("Сводка очистки по группам:")
    display(cleaning_summary)

    print(f"Файл сохранён: {cleaning_summary_path}")

else:
    print(
        "QC-поля ICA/AutoReject не найдены. "
        "Проверьте, что clean_epochs_artifacts() встроена в массовую обработку."
    )

In [ ]:
# @title 7.7. Проверка технической сопоставимости групп

technical_summary = (
    standardized_inventory
    .groupby("group")
    .agg(
        n_subjects=("subject_id", "nunique"),
        n_records=("record_id", "count"),
        n_epochs_total=("n_epochs", "sum"),
        sfreq_values=("sfreq_hz", lambda x: sorted(x.unique())),
        n_channels_values=("n_channels", lambda x: sorted(x.unique())),
        epoch_len_values=(
            "epoch_len_sec",
            lambda x: sorted(np.round(x.unique(), 6)),
        ),
    )
    .reset_index()
)

technical_summary_path = OUT["tables"] / "technical_summary_standardized.csv"
technical_summary.to_csv(technical_summary_path, index=False)

display(technical_summary)

logger.info(
    "Техническая сводка стандартизированных данных сохранена: %s",
    technical_summary_path,
)

### Итог этапа загрузки и стандартизации

В результате этапа загрузки и стандартизации данные ЧМТ и контрольной группы были приведены к единому формату MNE Epochs.

Итоговый стандартизированный набор данных включает:

- записи ЧМТ, загруженные из локальных `.mat`-файлов;
- контрольные записи, загруженные из локальных EDF-файлов OpenNeuro ds005385;
- единую частоту дискретизации;
- фиксированную длительность эпох;
- унифицированные названия каналов;
- сохранённую техническую информацию по каждой записи.

Следующий этап — унификация набора каналов между группами и контроль качества записей.


# Часть 8. Унификация набора каналов между группами

После загрузки и базовой стандартизации данных необходимо привести записи ЧМТ и контрольной группы к единому набору ЭЭГ-каналов.

Это необходимо по нескольким причинам:

1. данные ЧМТ и контрольной группы получены из разных источников;
2. исходные названия и состав каналов могут отличаться;
3. статистическое сравнение и машинное обучение требуют единого признакового пространства;
4. признаки по ROI должны рассчитываться на сопоставимых электродах;
5. сетевые метрики функциональной связности требуют одинакового набора узлов.

На этом этапе определяется пересечение каналов, доступных в обеих группах, после чего все записи приводятся к этому общему набору.



## 8.1. Проверка набора каналов после стандартизации

На этом этапе анализируется состав каналов в стандартизированных записях.

Для каждой записи ранее был сохранён список каналов в поле `channels`. Теперь эти списки используются для проверки:

- сколько каналов содержит каждая запись;
- какие каналы встречаются в группе ЧМТ;
- какие каналы встречаются в контрольной группе;
- какие каналы являются общими для обеих групп.

In [ ]:
# @title 8.1. Проверка списков каналов после стандартизации

def split_channel_string(channel_string: str) -> list[str]:
    """
    Преобразует строку с каналами в список названий каналов.

    Parameters
    ----------
    channel_string : str
        Строка с каналами, разделёнными символом "|".

    Returns
    -------
    list[str]
        Список названий каналов.
    """
    if pd.isna(channel_string):
        return []

    return [
        channel.strip()
        for channel in str(channel_string).split("|")
        if channel.strip()
    ]


def get_group_channel_sets(
    inventory: pd.DataFrame,
    group_name: str,
) -> list[set[str]]:
    """
    Возвращает список множеств каналов для записей одной группы.

    Parameters
    ----------
    inventory : pd.DataFrame
        Inventory стандартизированных записей.
    group_name : str
        Название группы.

    Returns
    -------
    list[set[str]]
        Список множеств каналов по записям.
    """
    group_inventory = inventory[inventory["group"] == group_name]

    channel_sets = [
        set(split_channel_string(channel_string))
        for channel_string in group_inventory["channels"]
    ]

    return channel_sets


tbi_channel_sets = get_group_channel_sets(
    inventory=standardized_inventory,
    group_name="TBI",
)

control_channel_sets = get_group_channel_sets(
    inventory=standardized_inventory,
    group_name="Control",
)

print("Проверка каналов после стандартизации")
print("-" * 60)
print(f"Число записей ЧМТ: {len(tbi_channel_sets)}")
print(f"Число записей контроля: {len(control_channel_sets)}")

print("\nЧисло каналов в записях ЧМТ:")
display(
    standardized_inventory
    .query("group == 'TBI'")
    ["n_channels"]
    .value_counts()
    .sort_index()
)

print("\nЧисло каналов в записях контроля:")
display(
    standardized_inventory
    .query("group == 'Control'")
    ["n_channels"]
    .value_counts()
    .sort_index()
)

## 8.2. Определение общего набора каналов

Для дальнейшего анализа используется только общий набор каналов, присутствующий во всех стандартизированных записях обеих групп.

Такой подход позволяет избежать ситуации, когда признаки в разных группах рассчитываются по разным электродам.

Общий набор каналов определяется как пересечение каналов по всем успешно обработанным записям ЧМТ и контрольной группы.

In [ ]:
# @title 8.2. Определение общего набора каналов

def intersect_channel_sets(channel_sets: list[set[str]]) -> set[str]:
    """
    Находит пересечение каналов по списку множеств.

    Parameters
    ----------
    channel_sets : list[set[str]]
        Список множеств каналов.

    Returns
    -------
    set[str]
        Пересечение каналов.
    """
    if not channel_sets:
        return set()

    common_channels = set(channel_sets[0])

    for channel_set in channel_sets[1:]:
        common_channels &= channel_set

    return common_channels


tbi_common_channels = intersect_channel_sets(tbi_channel_sets)
control_common_channels = intersect_channel_sets(control_channel_sets)

all_common_channels = tbi_common_channels & control_common_channels

# Для воспроизводимости сортируем каналы.
COMMON_CHANNELS = sorted(all_common_channels)

common_channels_path = OUT["tables"] / "common_channels.txt"

with open(common_channels_path, "w", encoding="utf-8") as file:
    for channel_name in COMMON_CHANNELS:
        file.write(f"{channel_name}\n")

print("Общий набор каналов")
print("-" * 60)
print(f"Каналов, общих для всех записей ЧМТ: {len(tbi_common_channels)}")
print(
    "Каналов, общих для всех записей контроля: "
    f"{len(control_common_channels)}"
)
print(f"Каналов, общих для обеих групп: {len(COMMON_CHANNELS)}")
print(f"Список сохранён: {common_channels_path}")

print("\nОбщие каналы:")
print(COMMON_CHANNELS)

In [ ]:
# @title 8.3. Сводка доступности каналов по группам

tbi_all_channels = sorted(set().union(*tbi_channel_sets))
control_all_channels = sorted(set().union(*control_channel_sets))

all_observed_channels = sorted(
    set(tbi_all_channels) | set(control_all_channels)
)

channel_availability = pd.DataFrame(
    {
        "channel": all_observed_channels,
    }
)

channel_availability["in_tbi"] = channel_availability["channel"].isin(
    tbi_all_channels
)

channel_availability["in_control"] = channel_availability["channel"].isin(
    control_all_channels
)

channel_availability["in_common_set"] = channel_availability["channel"].isin(
    COMMON_CHANNELS
)

channel_availability_path = OUT["tables"] / "channel_availability.csv"
channel_availability.to_csv(channel_availability_path, index=False)

print("Сводка доступности каналов:")
display(channel_availability)

logger.info(
    "Сводка доступности каналов сохранена: %s",
    channel_availability_path,
)

## 8.4. Применение общего набора каналов

После определения общего набора каналов все стандартизированные записи приводятся к этому набору.

Для каждой записи:

1. загружается сохранённый `.fif`-файл с эпохами;
2. выбираются только каналы из `COMMON_CHANNELS`;
3. порядок каналов приводится к единому;
4. запись сохраняется в отдельную папку `epochs_common_channels`.

Исходные стандартизированные файлы не перезаписываются.

In [ ]:
# @title  Применение общего набора каналов к сохранённым эпохам

COMMON_EPOCHS_DIR = OUTPUT_DIR / "epochs_common_channels"
COMMON_EPOCHS_DIR.mkdir(parents=True, exist_ok=True)


def apply_common_channels_to_epochs_file(
    epochs_path: Path,
    common_channels: list[str],
    output_dir: Path,
    group: str,
    record_id: str,
) -> str:
    """
    Загружает Epochs, оставляет общий набор каналов и сохраняет файл.

    Parameters
    ----------
    epochs_path : Path
        Путь к исходному .fif-файлу эпох.
    common_channels : list[str]
        Единый список каналов.
    output_dir : Path
        Папка для сохранения обновлённых эпох.
    group : str
        Группа данных.
    record_id : str
        Идентификатор записи.

    Returns
    -------
    str
        Путь к сохранённому файлу.
    """
    epochs = mne.read_epochs(
        epochs_path,
        preload=True,
        verbose=False,
    )

    # Сохраняем только общий набор каналов в едином порядке.
    epochs.pick(common_channels)

    group_output_dir = output_dir / group.lower()
    group_output_dir.mkdir(parents=True, exist_ok=True)

    safe_record_id = re.sub(r"[^A-Za-z0-9_.-]+", "_", str(record_id))
    output_path = group_output_dir / f"{safe_record_id}_common_epo.fif"

    epochs.save(
        output_path,
        overwrite=True,
        verbose=False,
    )

    return str(output_path)


common_channel_records = []

for _, row in tqdm(
    standardized_inventory.iterrows(),
    total=len(standardized_inventory),
    desc="Унификация каналов",
):
    epochs_path = row.get("epochs_path")

    if pd.isna(epochs_path) or epochs_path is None:
        register_processing_error(
            group=row["group"],
            file_path=Path(row["source_path"]),
            stage="common_channels",
            error=FileNotFoundError("Не указан путь к сохранённым эпохам."),
        )
        continue

    try:
        common_epochs_path = apply_common_channels_to_epochs_file(
            epochs_path=Path(epochs_path),
            common_channels=COMMON_CHANNELS,
            output_dir=COMMON_EPOCHS_DIR,
            group=row["group"],
            record_id=row["record_id"],
        )

        updated_row = row.to_dict()
        updated_row["common_epochs_path"] = common_epochs_path
        updated_row["n_common_channels"] = len(COMMON_CHANNELS)
        updated_row["common_channels"] = "|".join(COMMON_CHANNELS)

        common_channel_records.append(updated_row)

    except Exception as error:
        register_processing_error(
            group=row["group"],
            file_path=Path(row["source_path"]),
            stage="common_channels",
            error=error,
        )

common_channels_inventory = pd.DataFrame(common_channel_records)

common_channels_inventory_path = (
    OUT["tables"] / "inventory_epochs_common_channels.csv"
)

common_channels_inventory.to_csv(
    common_channels_inventory_path,
    index=False,
)

print("Унификация каналов завершена.")
print(f"Успешно обработано записей: {len(common_channels_inventory)}")
print(f"Число общих каналов: {len(COMMON_CHANNELS)}")
print(f"Файл сохранён: {common_channels_inventory_path}")

display(common_channels_inventory.head())

## 8.5. Проверка результата унификации каналов

После применения общего набора каналов проверяется, что все записи имеют одинаковое число каналов и одинаковый порядок каналов.

Это необходимо перед расчётом признаков, особенно для ROI-агрегации и сетевого анализа.

In [ ]:
# @title 8.5. Проверка результата унификации каналов

print("Проверка результата унификации каналов")
print("-" * 60)

print(f"Число записей после унификации: {len(common_channels_inventory)}")

print("\nЧисло записей по группам:")
display(
    common_channels_inventory
    .groupby("group")
    .agg(
        n_subjects=("subject_id", "nunique"),
        n_records=("record_id", "count"),
        n_epochs=("n_epochs", "sum"),
        median_epochs_per_record=("n_epochs", "median"),
        n_common_channels=("n_common_channels", "median"),
    )
    .reset_index()
)

print("\nРаспределение числа общих каналов:")
display(
    common_channels_inventory["n_common_channels"]
    .value_counts()
    .sort_index()
)

if common_channels_inventory["n_common_channels"].nunique() == 1:
    print("\nВсе записи имеют одинаковое число каналов.")
else:
    logger.warning("В записях обнаружено разное число каналов после унификации.")

# Часть 9. Контроль качества стандартизированных эпох

После унификации каналов необходимо выполнить базовый контроль качества записей и эпох.

На этом этапе проверяются технические характеристики данных:

- число успешно обработанных записей;
- число эпох в каждой записи;
- число каналов;
- частота дискретизации;
- длительность эпох;
- распределение амплитуд;
- наличие записей с аномально малым числом эпох;
- наличие потенциально артефактных записей.

Контроль качества нужен для того, чтобы исключить технически некорректные или явно нестабильные записи до расчёта признаков.

In [ ]:
# @title 9.1. Загрузка финальных inventory после cleaning-QC

control_epochs_inventory = pd.read_csv(
    OUT["tables"] / "control_epochs_inventory_analysis_ready.csv"
)

tbi_epochs_inventory = pd.read_csv(
    OUT["tables"] / "tbi_epochs_inventory_analysis_ready.csv"
)

cleaned_inventory = pd.concat(
    [control_epochs_inventory, tbi_epochs_inventory],
    ignore_index=True,
)

print("Финальные inventory после cleaning-QC")
print("-" * 70)
print(f"Контроль: {len(control_epochs_inventory)} записей")
print(f"ЧМТ: {len(tbi_epochs_inventory)} записей")
print(f"Всего: {len(cleaned_inventory)} записей")

In [ ]:
# @title 9.1  финальный контроль качества после ICA/ICLabel/AutoReject

# ---------------------------------------------------------------------
# 8.0. Параметры финального record-level QC
# ---------------------------------------------------------------------
# Эти критерии применяются одинаково к контрольной группе и группе ЧМТ.
# Запись исключается, если:
# 1. после очистки осталось слишком мало эпох;
# 2. AutoReject отклонил слишком большую долю эпох;
# 3. было выявлено слишком много плохих каналов.

MIN_EPOCHS_AFTER_CLEANING = 20
MAX_PROP_REJECTED_AUTOREJECT = 0.50
MAX_BAD_CHANNELS = 16

QC_DIR = OUT.get("quality_control", OUT["tables"].parent / "quality_control")
QC_DIR.mkdir(parents=True, exist_ok=True)

print("Параметры финального QC")
print("-" * 70)
print(f"Минимальное число эпох после очистки: {MIN_EPOCHS_AFTER_CLEANING}")
print(
    "Максимальная доля эпох, отклонённых AutoReject: "
    f"{MAX_PROP_REJECTED_AUTOREJECT}"
)
print(f"Максимальное число плохих каналов: {MAX_BAD_CHANNELS}")
print(f"Папка QC-таблиц: {QC_DIR}")


# ---------------------------------------------------------------------
# 8.1. Вспомогательные функции
# ---------------------------------------------------------------------
def load_cleaned_inventory(
    path: Path,
    group_name: str,
) -> pd.DataFrame:
    """
    Загружает cleaned inventory после массовой обработки.

    Parameters
    ----------
    path : Path
        Путь к inventory-файлу.
    group_name : str
        Название группы для вывода сообщений.

    Returns
    -------
    pd.DataFrame
        Загруженный inventory.
    """
    if not path.exists():
        raise FileNotFoundError(
            f"Не найден cleaned inventory для группы {group_name}: {path}. "
            "Сначала нужно завершить массовую обработку."
        )

    inventory = pd.read_csv(path)

    if inventory.empty:
        raise ValueError(
            f"Cleaned inventory для группы {group_name} пустой: {path}"
        )

    required_columns = [
        "group",
        "subject_id",
        "record_id",
        "n_epochs",
        "n_bad_channels",
        "n_removed_ica_components",
        "n_epochs_before_autoreject",
        "n_epochs_after_autoreject",
        "n_rejected_epochs_autoreject",
        "prop_rejected_epochs_autoreject",
        "epochs_path",
    ]

    missing_columns = [
        column for column in required_columns
        if column not in inventory.columns
    ]

    if missing_columns:
        raise ValueError(
            f"В cleaned inventory группы {group_name} отсутствуют колонки: "
            f"{missing_columns}"
        )

    print(f"{group_name}: inventory загружен")
    print(f"  файл: {path}")
    print(f"  записей: {len(inventory)}")
    print(f"  субъектов: {inventory['subject_id'].nunique()}")

    return inventory


def add_qc_flags(
    inventory: pd.DataFrame,
) -> pd.DataFrame:
    """
    Добавляет QC-флаги и причины исключения.

    Parameters
    ----------
    inventory : pd.DataFrame
        Inventory после массовой очистки.

    Returns
    -------
    pd.DataFrame
        Inventory с колонками qc_pass_cleaning и qc_exclusion_reason.
    """
    inventory = inventory.copy()

    inventory["qc_low_n_epochs"] = (
        inventory["n_epochs"] < MIN_EPOCHS_AFTER_CLEANING
    )

    inventory["qc_high_autoreject_rejection"] = (
        inventory["prop_rejected_epochs_autoreject"]
        > MAX_PROP_REJECTED_AUTOREJECT
    )

    inventory["qc_many_bad_channels"] = (
        inventory["n_bad_channels"] > MAX_BAD_CHANNELS
    )

    inventory["qc_pass_cleaning"] = (
        ~inventory["qc_low_n_epochs"]
        & ~inventory["qc_high_autoreject_rejection"]
        & ~inventory["qc_many_bad_channels"]
    )

    inventory["qc_exclusion_reason"] = ""

    inventory.loc[
        inventory["qc_low_n_epochs"],
        "qc_exclusion_reason",
    ] += "low_n_epochs;"

    inventory.loc[
        inventory["qc_high_autoreject_rejection"],
        "qc_exclusion_reason",
    ] += "high_autoreject_rejection;"

    inventory.loc[
        inventory["qc_many_bad_channels"],
        "qc_exclusion_reason",
    ] += "many_bad_channels;"

    return inventory


def summarize_cleaning_qc(
    inventory: pd.DataFrame,
) -> pd.DataFrame:
    """
    Формирует сводку QC по группе и статусу прохождения QC.

    Parameters
    ----------
    inventory : pd.DataFrame
        Inventory с QC-флагами.

    Returns
    -------
    pd.DataFrame
        QC-сводка.
    """
    summary = (
        inventory
        .groupby(["group", "qc_pass_cleaning"])
        .agg(
            n_subjects=("subject_id", "nunique"),
            n_records=("record_id", "count"),
            n_epochs=("n_epochs", "sum"),
            median_epochs=("n_epochs", "median"),
            min_epochs=("n_epochs", "min"),
            max_epochs=("n_epochs", "max"),
            median_bad_channels=("n_bad_channels", "median"),
            max_bad_channels=("n_bad_channels", "max"),
            median_removed_ica=("n_removed_ica_components", "median"),
            max_removed_ica=("n_removed_ica_components", "max"),
            median_prop_rejected_ar=(
                "prop_rejected_epochs_autoreject",
                "median",
            ),
            max_prop_rejected_ar=(
                "prop_rejected_epochs_autoreject",
                "max",
            ),
        )
        .reset_index()
    )

    return summary


def save_group_qc_outputs(
    inventory_with_qc: pd.DataFrame,
    output_prefix: str,
) -> tuple[pd.DataFrame, pd.DataFrame, pd.DataFrame]:
    """
    Сохраняет результаты QC для одной группы.

    Parameters
    ----------
    inventory_with_qc : pd.DataFrame
        Inventory с QC-флагами.
    output_prefix : str
        Префикс файлов: 'control' или 'tbi'.

    Returns
    -------
    tuple[pd.DataFrame, pd.DataFrame, pd.DataFrame]
        analysis-ready inventory, excluded inventory, QC summary.
    """
    ready_inventory = inventory_with_qc[
        inventory_with_qc["qc_pass_cleaning"]
    ].copy()

    excluded_inventory = inventory_with_qc[
        ~inventory_with_qc["qc_pass_cleaning"]
    ].copy()

    qc_summary = summarize_cleaning_qc(inventory_with_qc)

    inventory_with_qc_path = (
        OUT["tables"] / f"{output_prefix}_epochs_inventory_cleaned_with_qc.csv"
    )

    analysis_ready_path = (
        OUT["tables"] / f"{output_prefix}_epochs_inventory_analysis_ready.csv"
    )

    excluded_path = (
        QC_DIR / f"{output_prefix}_excluded_after_cleaning_qc.csv"
    )

    qc_summary_path = (
        QC_DIR / f"{output_prefix}_cleaning_qc_summary.csv"
    )

    inventory_with_qc.to_csv(inventory_with_qc_path, index=False)
    ready_inventory.to_csv(analysis_ready_path, index=False)
    excluded_inventory.to_csv(excluded_path, index=False)
    qc_summary.to_csv(qc_summary_path, index=False)

    print("\nСохранены QC-файлы:")
    print(f"  полный inventory с QC: {inventory_with_qc_path}")
    print(f"  analysis-ready inventory: {analysis_ready_path}")
    print(f"  исключённые записи: {excluded_path}")
    print(f"  QC summary: {qc_summary_path}")

    return ready_inventory, excluded_inventory, qc_summary


def display_excluded_records(
    excluded_inventory: pd.DataFrame,
) -> None:
    """
    Отображает исключённые записи в удобном виде.

    Parameters
    ----------
    excluded_inventory : pd.DataFrame
        Таблица исключённых записей.
    """
    if excluded_inventory.empty:
        print("Исключённых записей нет.")
        return

    excluded_columns = [
        "group",
        "subject_id",
        "record_id",
        "task",
        "acquisition",
        "n_epochs",
        "n_epochs_before_autoreject",
        "n_epochs_after_autoreject",
        "n_rejected_epochs_autoreject",
        "prop_rejected_epochs_autoreject",
        "n_bad_channels",
        "n_removed_ica_components",
        "qc_exclusion_reason",
        "epochs_path",
    ]

    existing_columns = [
        column for column in excluded_columns
        if column in excluded_inventory.columns
    ]

    display(
        excluded_inventory[existing_columns]
        .sort_values(
            [
                "prop_rejected_epochs_autoreject",
                "n_epochs",
            ],
            ascending=[False, True],
        )
    )


def run_group_cleaning_qc(
    cleaned_path: Path,
    group_name: str,
    output_prefix: str,
) -> tuple[pd.DataFrame, pd.DataFrame, pd.DataFrame, pd.DataFrame]:
    """
    Полный QC-процесс для одной группы.

    Parameters
    ----------
    cleaned_path : Path
        Путь к cleaned inventory.
    group_name : str
        Название группы для сообщений.
    output_prefix : str
        Префикс выходных файлов.

    Returns
    -------
    tuple[pd.DataFrame, pd.DataFrame, pd.DataFrame, pd.DataFrame]
        inventory с QC, analysis-ready, excluded, summary.
    """
    print("\n" + "=" * 80)
    print(f"Финальный QC группы {group_name}")
    print("=" * 80)

    inventory = load_cleaned_inventory(
        path=cleaned_path,
        group_name=group_name,
    )

    inventory_with_qc = add_qc_flags(inventory)

    ready_inventory, excluded_inventory, qc_summary = save_group_qc_outputs(
        inventory_with_qc=inventory_with_qc,
        output_prefix=output_prefix,
    )

    print("\nИтог QC")
    print("-" * 70)
    print(f"Всего записей: {len(inventory_with_qc)}")
    print(f"Прошли QC: {len(ready_inventory)}")
    print(f"Исключены: {len(excluded_inventory)}")

    display(qc_summary)

    if len(excluded_inventory) > 0:
        print("\nИсключённые записи:")
        display_excluded_records(excluded_inventory)

    return inventory_with_qc, ready_inventory, excluded_inventory, qc_summary


# ---------------------------------------------------------------------
# 8.2. Загрузка cleaned inventory и применение QC
# ---------------------------------------------------------------------
control_cleaned_path = OUT["tables"] / "control_epochs_inventory_cleaned.csv"
tbi_cleaned_path = OUT["tables"] / "tbi_epochs_inventory_cleaned.csv"

(
    control_epochs_inventory_with_qc,
    control_ready_after_cleaning,
    control_excluded_after_cleaning,
    control_qc_summary,
) = run_group_cleaning_qc(
    cleaned_path=control_cleaned_path,
    group_name="Control",
    output_prefix="control",
)

(
    tbi_epochs_inventory_with_qc,
    tbi_ready_after_cleaning,
    tbi_excluded_after_cleaning,
    tbi_qc_summary,
) = run_group_cleaning_qc(
    cleaned_path=tbi_cleaned_path,
    group_name="TBI",
    output_prefix="tbi",
)


# ---------------------------------------------------------------------
# 8.3. Общая сводка после финального QC
# ---------------------------------------------------------------------
analysis_ready_inventory_before_channels = pd.concat(
    [
        control_ready_after_cleaning,
        tbi_ready_after_cleaning,
    ],
    ignore_index=True,
)

analysis_ready_before_channels_path = (
    OUT["tables"] / "analysis_ready_inventory_before_common_channels.csv"
)

analysis_ready_inventory_before_channels.to_csv(
    analysis_ready_before_channels_path,
    index=False,
)

combined_cleaning_qc_summary = (
    analysis_ready_inventory_before_channels
    .groupby("group")
    .agg(
        n_subjects=("subject_id", "nunique"),
        n_records=("record_id", "count"),
        n_epochs=("n_epochs", "sum"),
        median_epochs=("n_epochs", "median"),
        min_epochs=("n_epochs", "min"),
        max_epochs=("n_epochs", "max"),
        median_bad_channels=("n_bad_channels", "median"),
        max_bad_channels=("n_bad_channels", "max"),
        median_removed_ica=("n_removed_ica_components", "median"),
        max_removed_ica=("n_removed_ica_components", "max"),
        median_prop_rejected_ar=(
            "prop_rejected_epochs_autoreject",
            "median",
        ),
        max_prop_rejected_ar=(
            "prop_rejected_epochs_autoreject",
            "max",
        ),
        median_sfreq=("sfreq_hz", "median"),
        median_epoch_len=("epoch_len_sec", "median"),
    )
    .reset_index()
)

combined_cleaning_qc_summary_path = (
    QC_DIR / "combined_cleaning_qc_summary_analysis_ready.csv"
)

combined_cleaning_qc_summary.to_csv(
    combined_cleaning_qc_summary_path,
    index=False,
)

print("\n" + "=" * 80)
print("Общая сводка analysis-ready набора после финального QC")
print("=" * 80)

display(combined_cleaning_qc_summary)

print("\nAnalysis-ready inventory до унификации каналов сохранён:")
print(analysis_ready_before_channels_path)

print("\nОбщая QC-сводка сохранена:")
print(combined_cleaning_qc_summary_path)


# ---------------------------------------------------------------------
# 8.4. Контроль ожидаемых чисел записей
# ---------------------------------------------------------------------
expected_control_records = 363
expected_tbi_records = 212
expected_total_records = expected_control_records + expected_tbi_records

actual_control_records = len(control_ready_after_cleaning)
actual_tbi_records = len(tbi_ready_after_cleaning)
actual_total_records = len(analysis_ready_inventory_before_channels)

print("\nПроверка ожидаемых размеров analysis-ready набора")
print("-" * 70)
print(f"Control: {actual_control_records} / expected {expected_control_records}")
print(f"TBI: {actual_tbi_records} / expected {expected_tbi_records}")
print(f"Total: {actual_total_records} / expected {expected_total_records}")

if actual_control_records != expected_control_records:
    logger.warning(
        "Ожидалось %d записей контроля после QC, получено %d.",
        expected_control_records,
        actual_control_records,
    )

if actual_tbi_records != expected_tbi_records:
    logger.warning(
        "Ожидалось %d записей ЧМТ после QC, получено %d.",
        expected_tbi_records,
        actual_tbi_records,
    )

if actual_total_records != expected_total_records:
    logger.warning(
        "Ожидалось %d записей всего после QC, получено %d.",
        expected_total_records,
        actual_total_records,
    )

print("\nФинальный QC завершён.")

In [ ]:
# @title 9.2. Амплитудный контроль качества после ICA/ICLabel/AutoReject

def compute_epochs_amplitude_qc(
    inventory: pd.DataFrame,
    p2p_threshold_uv: float = 300.0,
) -> pd.DataFrame:
    """
    Рассчитывает амплитудный QC по очищенным эпохам.

    Parameters
    ----------
    inventory : pd.DataFrame
        Analysis-ready inventory после cleaning-QC.
    p2p_threshold_uv : float
        Порог peak-to-peak амплитуды эпохи в мкВ.

    Returns
    -------
    pd.DataFrame
        Таблица амплитудного QC по записям.
    """
    records = []

    for _, row in tqdm(
        inventory.iterrows(),
        total=len(inventory),
        desc="Амплитудный QC",
    ):
        epochs_path = Path(row["epochs_path"])

        try:
            epochs = mne.read_epochs(
                epochs_path,
                preload=True,
                verbose=False,
            )

            data_uv = epochs.get_data(copy=True) * 1e6

            # Peak-to-peak по каналам внутри эпохи, затем максимум по каналам.
            epoch_p2p_uv = np.ptp(data_uv, axis=2).max(axis=1)

            n_bad_epochs = int(np.sum(epoch_p2p_uv > p2p_threshold_uv))
            prop_bad_epochs = n_bad_epochs / len(epoch_p2p_uv)

            records.append(
                {
                    "group": row["group"],
                    "subject_id": row["subject_id"],
                    "record_id": row["record_id"],
                    "n_epochs": len(epoch_p2p_uv),
                    "p2p_threshold_uv": p2p_threshold_uv,
                    "p2p_median_uv": float(np.median(epoch_p2p_uv)),
                    "p2p_p95_uv": float(np.percentile(epoch_p2p_uv, 95)),
                    "p2p_max_uv": float(np.max(epoch_p2p_uv)),
                    "n_bad_epochs_p2p": n_bad_epochs,
                    "prop_bad_epochs_p2p": float(prop_bad_epochs),
                    "epochs_path": str(epochs_path),
                }
            )

            del epochs
            gc.collect()

        except Exception as error:
            register_processing_error(
                group=row.get("group", "unknown"),
                file_path=epochs_path,
                stage="amplitude_qc_after_cleaning",
                error=error,
            )

    return pd.DataFrame(records)


P2P_THRESHOLD_UV = 300.0

amplitude_qc = compute_epochs_amplitude_qc(
    inventory=analysis_ready_inventory_before_channels,
    p2p_threshold_uv=P2P_THRESHOLD_UV,
)

QC_DIR = OUT.get("quality_control", OUT["tables"].parent / "quality_control")
QC_DIR.mkdir(parents=True, exist_ok=True)

amplitude_qc_path = QC_DIR / "amplitude_qc_after_cleaning.csv"

amplitude_qc.to_csv(
    amplitude_qc_path,
    index=False,
)

print("Амплитудный QC после очистки завершён.")
print(f"Файл сохранён: {amplitude_qc_path}")

display(
    amplitude_qc
    .groupby("group")
    .agg(
        n_records=("record_id", "count"),
        median_p2p_uv=("p2p_median_uv", "median"),
        median_p95_p2p_uv=("p2p_p95_uv", "median"),
        max_p2p_uv=("p2p_max_uv", "max"),
        median_prop_bad_epochs=("prop_bad_epochs_p2p", "median"),
        max_prop_bad_epochs=("prop_bad_epochs_p2p", "max"),
    )
    .reset_index()
)

In [ ]:
# @title 9.2.1. Записи с высокой долей эпох выше p2p-порога

BAD_EPOCH_PROP_WARNING = 0.30
BAD_EPOCH_PROP_EXCLUDE = 0.50

QC_DIR = OUT.get("quality_control", OUT["tables"].parent / "quality_control")
QC_DIR.mkdir(parents=True, exist_ok=True)

high_artifact_records = amplitude_qc[
    amplitude_qc["prop_bad_epochs_p2p"] >= BAD_EPOCH_PROP_WARNING
].copy()

high_artifact_records["amplitude_qc_level"] = np.where(
    high_artifact_records["prop_bad_epochs_p2p"] >= BAD_EPOCH_PROP_EXCLUDE,
    "exclude_candidate",
    "warning",
)

high_artifact_path = (
    QC_DIR / "high_artifact_records_amplitude_qc_after_cleaning.csv"
)

high_artifact_records.to_csv(
    high_artifact_path,
    index=False,
)

print("Записи с высокой долей эпох выше p2p-порога")
print("-" * 70)
print(f"Порог предупреждения: {BAD_EPOCH_PROP_WARNING}")
print(f"Порог исключения-кандидата: {BAD_EPOCH_PROP_EXCLUDE}")
print(f"Найдено записей: {len(high_artifact_records)}")
print(f"Файл сохранён: {high_artifact_path}")

if len(high_artifact_records) > 0:
    display(
        high_artifact_records
        .sort_values("prop_bad_epochs_p2p", ascending=False)
        .head(30)
    )
else:
    print("Записей с высокой долей p2p-артефактных эпох не обнаружено.")

In [ ]:
# @title 9.2.2 Визуализация амплитудного QC

fig, axes = plt.subplots(1, 2, figsize=(15, 5))

for group_name, color in GROUP_COLORS.items():
    group_df = amplitude_qc[amplitude_qc["group"] == group_name]

    axes[0].hist(
        group_df["p2p_median_uv"].dropna(),
        bins=30,
        color=color,
        label=GROUP_LABELS_RU[group_name],
        alpha=1.0,
    )

axes[0].axvline(
    P2P_THRESHOLD_UV,
    color="black",
    linestyle="--",
    linewidth=1.5,
    label=f"Порог {P2P_THRESHOLD_UV:.0f} мкВ",
)
axes[0].set_title("Медианная p2p-амплитуда после очистки")
axes[0].set_xlabel("p2p-амплитуда, мкВ")
axes[0].set_ylabel("Количество записей")

for group_name, color in GROUP_COLORS.items():
    group_df = amplitude_qc[amplitude_qc["group"] == group_name]

    axes[1].hist(
        group_df["prop_bad_epochs_p2p"].dropna(),
        bins=30,
        color=color,
        label=GROUP_LABELS_RU[group_name],
        alpha=1.0,
    )

axes[1].axvline(
    BAD_EPOCH_PROP_WARNING,
    color="black",
    linestyle="--",
    linewidth=1.5,
    label="Предупреждение",
)
axes[1].axvline(
    BAD_EPOCH_PROP_EXCLUDE,
    color="black",
    linestyle=":",
    linewidth=1.5,
    label="Кандидат на исключение",
)
axes[1].set_title("Доля эпох выше p2p-порога")
axes[1].set_xlabel("Доля эпох")
axes[1].set_ylabel("Количество записей")

handles, labels = axes[1].get_legend_handles_labels()
fig.legend(
    handles,
    labels,
    loc="lower center",
    ncol=4,
    frameon=False,
    bbox_to_anchor=(0.5, -0.03),
)

fig.suptitle("Амплитудный контроль качества после очистки", fontsize=16)
fig.tight_layout()

figure_path = OUT["qc"] / "amplitude_qc_after_cleaning.png"
save_figure(fig, figure_path)

plt.show()

In [ ]:
# @title 9.3. Визуализация состава analysis-ready набора

plot_df = analysis_ready_inventory_before_channels.copy()

summary = (
    plot_df
    .groupby("group")
    .agg(
        n_subjects=("subject_id", "nunique"),
        n_records=("record_id", "count"),
        n_epochs=("n_epochs", "sum"),
    )
    .reset_index()
)

fig, axes = plt.subplots(1, 3, figsize=(16, 5))

metrics = [
    ("n_subjects", "Число субъектов"),
    ("n_records", "Число записей"),
    ("n_epochs", "Число эпох"),
]

for ax, (metric, title) in zip(axes, metrics):
    values = summary.set_index("group").loc[["TBI", "Control"], metric]
    labels = ["ЧМТ", "Контроль"]
    colors = [COL_TBI, COL_CONTROL]

    ax.bar(labels, values, color=colors)
    ax.set_title(title)
    ax.set_ylabel("Количество")

    for i, value in enumerate(values):
        ax.text(
            i,
            value,
            f"{int(value)}",
            ha="center",
            va="bottom",
            fontsize=12,
        )

fig.suptitle("Состав analysis-ready набора после финального QC", fontsize=16)
fig.tight_layout()

figure_path = OUT["qc"] / "sample_composition_after_cleaning_qc.png"
save_figure(fig, figure_path)

plt.show()

display(summary)

In [ ]:
# @title 9.4. Распределение числа эпох после финального QC

fig, ax = plt.subplots(figsize=(10, 6))

for group_name, color in GROUP_COLORS.items():
    group_df = analysis_ready_inventory_before_channels[
        analysis_ready_inventory_before_channels["group"] == group_name
    ]

    ax.hist(
        group_df["n_epochs"],
        bins=30,
        color=color,
        label=GROUP_LABELS_RU[group_name],
        alpha=1.0,
    )

ax.axvline(
    MIN_EPOCHS_AFTER_CLEANING,
    color="black",
    linestyle="--",
    linewidth=1.5,
    label=f"QC-порог: {MIN_EPOCHS_AFTER_CLEANING} эпох",
)

ax.set_title("Распределение числа эпох после финального QC")
ax.set_xlabel("Число 4-секундных эпох")
ax.set_ylabel("Количество записей")
ax.legend(frameon=False)

figure_path = OUT["qc"] / "n_epochs_after_cleaning_qc.png"
save_figure(fig, figure_path)

plt.show()

In [ ]:
# @title 9.5. Распределение QC-показателей очистки

qc_metrics = [
    ("n_bad_channels", "Число плохих каналов"),
    ("n_removed_ica_components", "Число удалённых ICA-компонент"),
    ("prop_rejected_epochs_autoreject", "Доля эпох, отклонённых AutoReject"),
]

fig, axes = plt.subplots(1, 3, figsize=(18, 5))

for ax, (metric, title) in zip(axes, qc_metrics):
    for group_name, color in GROUP_COLORS.items():
        group_df = analysis_ready_inventory_before_channels[
            analysis_ready_inventory_before_channels["group"] == group_name
        ]

        ax.hist(
            group_df[metric].dropna(),
            bins=25,
            color=color,
            label=GROUP_LABELS_RU[group_name],
            alpha=1.0,
        )

    ax.set_title(title)
    ax.set_xlabel(title)
    ax.set_ylabel("Количество записей")

axes[0].axvline(
    MAX_BAD_CHANNELS,
    color="black",
    linestyle="--",
    linewidth=1.5,
)

axes[2].axvline(
    MAX_PROP_REJECTED_AUTOREJECT,
    color="black",
    linestyle="--",
    linewidth=1.5,
)

handles, labels = axes[0].get_legend_handles_labels()
fig.legend(
    handles,
    labels,
    loc="lower center",
    ncol=2,
    frameon=False,
    bbox_to_anchor=(0.5, -0.03),
)

fig.suptitle("QC-показатели после ICA/ICLabel и AutoReject", fontsize=16)
fig.tight_layout()

figure_path = OUT["qc"] / "cleaning_qc_metrics_after_final_qc.png"
save_figure(fig, figure_path)

plt.show()

In [ ]:
# @title 9.6. Проверка унификации каналов

channel_summary = (
    common_channels_inventory
    .groupby(["group", "n_common_channels"])
    .agg(
        n_subjects=("subject_id", "nunique"),
        n_records=("record_id", "count"),
        n_epochs=("n_epochs", "sum"),
    )
    .reset_index()
)

print("Проверка унификации каналов")
print("-" * 70)
display(channel_summary)

fig, ax = plt.subplots(figsize=(8, 5))

for group_name, color in GROUP_COLORS.items():
    group_df = channel_summary[channel_summary["group"] == group_name]

    ax.bar(
        group_df["n_common_channels"].astype(str),
        group_df["n_records"],
        color=color,
        label=GROUP_LABELS_RU[group_name],
        alpha=1.0,
    )

ax.set_title("Число каналов после унификации")
ax.set_xlabel("Число общих каналов")
ax.set_ylabel("Количество записей")
ax.legend(frameon=False)

figure_path = OUT["qc"] / "common_channels_after_unification.png"
save_figure(fig, figure_path)

plt.show()

In [ ]:
# @title 9.7. Сводка амплитудного QC по группам

amplitude_qc_summary = (
    amplitude_qc
    .groupby("group")
    .agg(
        n_records=("record_id", "count"),
        median_p2p_uv=("p2p_median_uv", "median"),
        median_p95_p2p_uv=("p2p_p95_uv", "median"),
        median_prop_bad_epochs=("prop_bad_epochs_p2p", "median"),
        max_prop_bad_epochs=("prop_bad_epochs_p2p", "max"),
    )
    .reset_index()
)

amplitude_qc_summary_path = OUT["qc"] / "amplitude_qc_summary_by_group.csv"
amplitude_qc_summary.to_csv(amplitude_qc_summary_path, index=False)

print("Сводка амплитудного QC по группам:")
display(amplitude_qc_summary)

logger.info(
    "Сводка амплитудного QC сохранена: %s",
    amplitude_qc_summary_path,
)

In [ ]:
# @title 9.8. Формирование итоговой таблицы analysis-ready записей с amplitude-QC аудитом

# ---------------------------------------------------------------------
# Назначение ячейки
# ---------------------------------------------------------------------
# На этом этапе финальный набор записей уже прошёл:
# 1. ICA/ICLabel;
# 2. AutoReject;
# 3. record-level cleaning-QC;
# 4. унификацию каналов.
#
# Поэтому amplitude-QC используется как дополнительный аудит качества,
# а не как новый автоматический критерий исключения записей.
# ---------------------------------------------------------------------

id_columns = ["group", "subject_id", "record_id"]

QC_DIR = OUT.get("quality_control", OUT["tables"].parent / "quality_control")
QC_DIR.mkdir(parents=True, exist_ok=True)

# ---------------------------------------------------------------------
# Проверка входных таблиц
# ---------------------------------------------------------------------
required_common_columns = id_columns + [
    "n_epochs",
    "n_common_channels",
    "common_epochs_path",
]

missing_common_columns = [
    column for column in required_common_columns
    if column not in common_channels_inventory.columns
]

if missing_common_columns:
    raise ValueError(
        "В common_channels_inventory отсутствуют необходимые колонки: "
        f"{missing_common_columns}"
    )

required_amplitude_columns = id_columns + [
    "prop_bad_epochs_p2p",
    "p2p_median_uv",
    "p2p_p95_uv",
    "p2p_max_uv",
]

missing_amplitude_columns = [
    column for column in required_amplitude_columns
    if column not in amplitude_qc.columns
]

if missing_amplitude_columns:
    raise ValueError(
        "В amplitude_qc отсутствуют необходимые колонки: "
        f"{missing_amplitude_columns}"
    )

# ---------------------------------------------------------------------
# Подготовка amplitude_qc к merge
# ---------------------------------------------------------------------
amplitude_qc_for_merge = amplitude_qc.copy()

# В amplitude_qc может быть epochs_path, а в common_channels_inventory —
# common_epochs_path. Для merge достаточно group/subject_id/record_id.
columns_to_drop_before_merge = [
    "n_epochs",
    "epochs_path",
    "common_epochs_path",
]

amplitude_qc_for_merge = amplitude_qc_for_merge.drop(
    columns=columns_to_drop_before_merge,
    errors="ignore",
)

amplitude_qc_for_merge = amplitude_qc_for_merge.drop_duplicates(
    subset=id_columns,
    keep="last",
)

# ---------------------------------------------------------------------
# Объединение финального inventory с amplitude-QC показателями
# ---------------------------------------------------------------------
analysis_inventory = common_channels_inventory.merge(
    amplitude_qc_for_merge,
    on=id_columns,
    how="left",
    validate="one_to_one",
)

# ---------------------------------------------------------------------
# Флаги аудита качества
# ---------------------------------------------------------------------
BAD_EPOCH_PROP_WARNING = 0.30
BAD_EPOCH_PROP_EXCLUDE_CANDIDATE = 0.50

analysis_inventory["amplitude_qc_warning"] = (
    analysis_inventory["prop_bad_epochs_p2p"]
    >= BAD_EPOCH_PROP_WARNING
)

analysis_inventory["amplitude_qc_exclude_candidate"] = (
    analysis_inventory["prop_bad_epochs_p2p"]
    >= BAD_EPOCH_PROP_EXCLUDE_CANDIDATE
)

analysis_inventory["amplitude_qc_note"] = ""

analysis_inventory.loc[
    analysis_inventory["amplitude_qc_warning"],
    "amplitude_qc_note",
] += "high_p2p_epoch_fraction_warning;"

analysis_inventory.loc[
    analysis_inventory["amplitude_qc_exclude_candidate"],
    "amplitude_qc_note",
] += "high_p2p_epoch_fraction_exclude_candidate;"

# ---------------------------------------------------------------------
# Финальный флаг пригодности
# ---------------------------------------------------------------------
# Записи уже были отфильтрованы в разделе cleaning-QC.
# Поэтому здесь qc_pass фиксирует, что запись входит в финальный
# analysis-ready набор после унификации каналов.
analysis_inventory["qc_pass"] = True

# ---------------------------------------------------------------------
# Сохранение итоговых таблиц
# ---------------------------------------------------------------------
analysis_inventory_path = (
    QC_DIR / "analysis_inventory_with_qc.csv"
)

analysis_ready_inventory_path = (
    OUT["tables"] / "analysis_ready_inventory_cleaned.csv"
)

analysis_inventory_path.parent.mkdir(parents=True, exist_ok=True)
analysis_ready_inventory_path.parent.mkdir(parents=True, exist_ok=True)

analysis_inventory.to_csv(
    analysis_inventory_path,
    index=False,
)

analysis_ready_inventory_cleaned = analysis_inventory.copy()

analysis_ready_inventory_cleaned.to_csv(
    analysis_ready_inventory_path,
    index=False,
)

# Для совместимости со старыми ячейками можно дополнительно сохранить
# копию под старым именем, но основным считать файл с суффиксом _cleaned.
legacy_analysis_ready_path = (
    OUT["tables"] / "analysis_ready_inventory.csv"
)

analysis_ready_inventory_cleaned.to_csv(
    legacy_analysis_ready_path,
    index=False,
)

# ---------------------------------------------------------------------
# Сводки для проверки
# ---------------------------------------------------------------------
print("Формирование итоговой таблицы analysis-ready записей завершено.")
print("-" * 70)
print(f"Всего записей: {len(analysis_inventory)}")
print(f"Файл с QC-аудитом сохранён: {analysis_inventory_path}")
print(f"Финальный файл для анализа сохранён: {analysis_ready_inventory_path}")
print(f"Копия для совместимости сохранена: {legacy_analysis_ready_path}")

print("\nСводка по группам:")
display(
    analysis_inventory
    .groupby("group")
    .agg(
        n_subjects=("subject_id", "nunique"),
        n_records=("record_id", "count"),
        n_epochs=("n_epochs", "sum"),
        median_epochs=("n_epochs", "median"),
        min_epochs=("n_epochs", "min"),
        max_epochs=("n_epochs", "max"),
        n_common_channels=("n_common_channels", "median"),
        median_prop_bad_epochs_p2p=("prop_bad_epochs_p2p", "median"),
        max_prop_bad_epochs_p2p=("prop_bad_epochs_p2p", "max"),
        n_amplitude_warnings=("amplitude_qc_warning", "sum"),
        n_amplitude_exclude_candidates=(
            "amplitude_qc_exclude_candidate",
            "sum",
        ),
    )
    .reset_index()
)

print("\nПроверка числа каналов:")
display(
    analysis_inventory["n_common_channels"]
    .value_counts()
    .sort_index()
)

print("\nПроверка пропусков amplitude-QC после merge:")
display(
    analysis_inventory
    .groupby("group")
    .agg(
        n_records=("record_id", "count"),
        missing_prop_bad_epochs_p2p=("prop_bad_epochs_p2p", lambda x: x.isna().sum()),
        missing_p2p_median=("p2p_median_uv", lambda x: x.isna().sum()),
    )
    .reset_index()
)

print("\nЗаписи с amplitude-QC предупреждениями:")
amplitude_warning_records = analysis_inventory[
    analysis_inventory["amplitude_qc_warning"]
].copy()

print(f"Найдено записей с предупреждениями: {len(amplitude_warning_records)}")

if len(amplitude_warning_records) > 0:
    warning_columns = [
        "group",
        "subject_id",
        "record_id",
        "n_epochs",
        "n_common_channels",
        "p2p_median_uv",
        "p2p_p95_uv",
        "p2p_max_uv",
        "prop_bad_epochs_p2p",
        "amplitude_qc_note",
        "common_epochs_path",
    ]

    existing_warning_columns = [
        column for column in warning_columns
        if column in amplitude_warning_records.columns
    ]

    display(
        amplitude_warning_records[existing_warning_columns]
        .sort_values("prop_bad_epochs_p2p", ascending=False)
        .head(30)
    )
else:
    print("Записей с amplitude-QC предупреждениями не обнаружено.")

In [ ]:
# @title 9.10. Сохранение таблицы ошибок обработки

# ---------------------------------------------------------------------
# Назначение ячейки
# ---------------------------------------------------------------------
# В течение массовой обработки ошибки сохранялись в список processing_errors.
# Здесь этот список финально сохраняется в CSV-файл для аудита пайплайна.
# Если ошибок нет, создаётся пустая таблица с ожидаемыми колонками.
# ---------------------------------------------------------------------

ERRORS_DIR = OUT.get("errors", OUT["tables"].parent / "errors")
ERRORS_DIR.mkdir(parents=True, exist_ok=True)

processing_errors_path = ERRORS_DIR / "processing_errors.csv"

error_columns = [
    "group",
    "file_path",
    "stage",
    "error_type",
    "error_message",
    "timestamp",
]

if len(processing_errors) > 0:
    processing_errors_df = pd.DataFrame(processing_errors)

    # Приводим названия колонок к более явному виду, если ошибки
    # сохранялись в старом формате.
    rename_map = {
        "error": "error_message",
        "message": "error_message",
        "path": "file_path",
    }

    processing_errors_df = processing_errors_df.rename(
        columns=rename_map,
    )

    for column in error_columns:
        if column not in processing_errors_df.columns:
            processing_errors_df[column] = np.nan

    # Сохраняем все дополнительные колонки тоже, но основные ставим первыми.
    other_columns = [
        column for column in processing_errors_df.columns
        if column not in error_columns
    ]

    processing_errors_df = processing_errors_df[
        error_columns + other_columns
    ]

else:
    processing_errors_df = pd.DataFrame(columns=error_columns)

processing_errors_df.to_csv(
    processing_errors_path,
    index=False,
)

print("Проверка ошибок обработки завершена.")
print("-" * 70)
print(f"Число зарегистрированных ошибок: {len(processing_errors_df)}")
print(f"Файл ошибок сохранён: {processing_errors_path}")

if len(processing_errors_df) > 0:
    print("\nСводка ошибок по этапам:")
    display(
        processing_errors_df
        .groupby(["group", "stage"])
        .size()
        .reset_index(name="n_errors")
        .sort_values("n_errors", ascending=False)
    )

    print("\nПервые строки таблицы ошибок:")
    display(processing_errors_df.head(20))
else:
    print("Ошибок обработки не зарегистрировано.")